# Sales Brief 郵件報告 Job（前線業務快報）v2

v2（2026-10-27，依 `review_202610_questions.md` 修正）與 v1 的差異：

| 項目 | v1 | v2 |
|------|----|----|
| 品牌名單 | 只抓當月已送出的品牌（NB 202610 少了 Dell / Acer） | 近 `brand_lookback_months`（預設 6）個月內有送出年度 BP 的品牌；`APP_CONFIG.brand_order` 可固定 |
| 缺填品牌 | 各區塊處理不一致（1a 整列 `-`、1b/2b 用舊值） | 統一「沿用最近一次填報」並標 `＊`；頂部紅色警示列出缺填品牌；Section 4 顯示「（Oct'26 未填報）」；`on_missing_brand=fail` 可改為該 app 不產出 |
| 季度未填齊 | 只加到部分月份就顯示（26 4Q=1.4、QoQ -42%） | 三個月未齊 → `-`，不計入 Major，註解列出 |
| Major MoM | 直接加總（某品牌缺最後一月 → Dec -52%） | 每月只加總前後兩月都有數字的品牌（同口徑） |
| 當月 (F) | `period > fill_period`，當月不標 | `>=`，當月即 (F)（與 YTD「不含當月」同定義）；附件 SVG 預估段改虛線＋灰底 |
| 品牌配色 | 依排名位置取色（排名變就換色） | `APP_CONFIG.brand_colors` 固定，以 202609_2 已寄出版為準 |
| Y26 (Pre) | `target_month = fill_month-1` 的最新一筆（會被之後補送蓋掉） | 該品牌 `as_of` 之前一期、且 target 期＝填報期的那筆 |
| LLM | 無後處理；prompt 無年份、近三月無月份標籤 | `clean_insight` 去 markdown / 重複標題；prompt 帶報告月份、欄位定義、月份標籤、(F)/沿用/`-` 意義 |
| 取數 | `generate_html` 與 c06 各取一次 | `build_report_context` 取一次，三欄共用 ctx |
| 關卡 | 只檢查格式 | c06 `content_checks` + c07 內容關卡（品牌數比上期少、(F)、配色與上期一致、markdown、「明年」、缺填 / 未填齊）→ ⚠️ 不擋寫入 |


每次執行都會重新產出 TV / NB / MNT 各應用**最新完整月份**的報告，以 MERGE 寫入目標表；
下游 myAgent 讀表、畫 PNG、寄信。

- 來源（`micenter.mi3_datahub_prod`）：`g_sales_brief_numeric_entry`（`status='submitted'`）、
  `g_sales_brief_brand_summary`、`g_sales_brief_app_summary`；三表 period 交集的最新月份即該 app 的產出月份
- 目標：`micenter.mi3_datahub_prod.g_sales_brief_mail_report`，key＝`(app_name, brief_month)`，歷史月份保留
- LLM：Azure Foundry `claude-sonnet-5`（secret `llm-api-keys/ms-foundry`），只產 AI Insight 文字，數字與 HTML 一律由程式填入

## 交付三欄

| 欄位 | 內容 | 消費方式 |
|------|------|----------|
| `html_content` | 完整圖表版 HTML（含 SVG 折線圖） | 落地為信件 .html 附件 |
| `mail_body` | Outlook 相容 640px table 版、minified；圖表處為 `{{CHART1_PNG_URL}}` / `{{CHART2_PNG_URL}}` 佔位符 | 替換佔位符後當信件內文 |
| `chart_data` | 兩組圖表數據 JSON（schema 見下） | myAgent 畫成 PNG |

三欄同源：`html_content` 的 SVG 與 `chart_data` 吃同一份中間結構；`mail_body` 沿用 `html_content` 已產生的 AI Insight（`_INSIGHT_CACHE`），不重呼叫 LLM。

## Cell 流程

| Cell | 內容 |
|------|------|
| `[c01] install` | `%pip install anthropic httpx`（job cluster 未裝 library 時才需要） |
| `[c02] source_check` | widgets `on_missing_brand` / `brand_lookback_months`；三表交集取各 app 最新月份；品牌填報完整度檢查 → `apps_to_process` |
| `[c03] helpers` | `APP_CONFIG`（含 `brand_colors`）、標題 / 連結、燈號、缺填標記、Major 合計（含同口徑 MoM）、`clean_insight`、取數函式、圖表中間結構、`content_checks` |
| `[c04] llm_init` | 覆寫 `call_claude` 為實際 Foundry 呼叫 |
| `[c05] render` | 吃 ctx：`generate_insights` / `generate_html` / `generate_mail_body` / `generate_chart_data`；SVG 共用 `_svg_line_chart` |
| `[c06] run_merge` | `build_report_context` 取數一次 → 三欄 → `content_checks` 警告 → MERGE 寫入目標表 |
| `[c07] validate` | 讀回目標表做格式關卡（❌）與內容關卡（⚠️），並與上一期比對 |
| `[c08]`–`[c10]` | 開發用：預覽 MNT / TV / NB 最新一筆三欄 |

## 報告格式

- 標題：`DSBG MI3.0 | {Mon'YY} {Monitor | TV | NB} 前線業務快報`（月份＝產出月份；名稱取 `APP_CONFIG[app]["display_name"]`）
- 開頭連結（header 內）：進入 MI3.0 系統對應頁面，URL 取 `APP_CONFIG[app]["dashboard_url"]`

| App | 系統頁面 |
|-----|----------|
| MNT | `https://mi30.azurewebsites.net/shell/reportdashboard?groupId=75` |
| TV | `https://mi30.azurewebsites.net/shell/reportdashboard?groupId=102` |
| NB | `https://mi30.azurewebsites.net/shell/reportdashboard?groupId=95` |

- 庫存燈號（Section 5b，設定在 `APP_CONFIG[app]["inv_thresholds"]`）：

| App | 綠燈 | 黃燈 | 紅燈 |
|-----|------|------|------|
| MNT | < 12w | ≥ 12w | ≥ 20w（ASUS：黃 ≥ 20w、紅 ≥ 24w） |
| NB | 3w ~ 5w | ≥ 5w | ≥ 10w（偏高）或 < 3w（過低） |
| TV — Samsung | ≤ 14w | > 14w | ≥ 16w |
| TV — Hisense / TCL / TPV | ≤ 7w | > 7w | ≥ 10w |

  週數先四捨五入成整數（`round_weeks`），表格顯示與燈號判斷都用這個值。TV 黃燈為嚴格大於（`inv_yellow_strict`），MNT / NB 為大於等於。

## `chart_data` schema（job 與 myAgent 的契約，改動需先同步下游）

```json
{
  "app_name": "MNT",
  "brief_month": "2026-09",
  "charts": [
    {
      "id": "CHART1",
      "placeholder": "{{CHART1_PNG_URL}}",
      "chart_type": "line",
      "title": "四品牌月度面板採購趨勢（MI3.0）",
      "unit": "M pcs",
      "x_labels": ["25-Jan", "...", "Nov(F)"],
      "forecast_from_index": 17,
      "series": [{ "name": "Dell", "color": "#2a78d6", "data": [2.4, 1.5, null] }]
    },
    { "id": "CHART2", "placeholder": "{{CHART2_PNG_URL}}", "unit": "week", "forecast_from_index": null, "...": "同上" }
  ]
}
```

- `charts` 數量＝`mail_body` 佔位符數；`placeholder` 逐字對應 `mail_body` 內字串
- `forecast_from_index`：該索引（含）之後為預測值（虛線）；**當月即為預測值**（本期填報、尚未結算，與 YTD 不含當月同定義）；無預測為 `null`
- `series` 只放品牌排序前 4 名（NB 依年度 BP 排序，Acer / MSI 不入信件圖）；附件 SVG 最多畫 6 個品牌。兩者數量不同為刻意設計，已與使用者確認
- 缺值為 `null`，畫圖跳點不補 0；色碼依品牌固定（`APP_CONFIG[app]["brand_colors"]`），與附件 SVG 相同

## 品質關卡（`[c07]`）

| 項目 | 規則 |
|------|------|
| 佔位符 | `mail_body` 恰含 CHART1 / CHART2 各一次、無其他 `{{…}}`；`html_content` 無 `{{…}}` |
| 體積 | `mail_body` ≤ 35,000 字元（下游寄信內文上限的保守值） |
| chart JSON | 可解析；每組 `series[].data` 長度＝`x_labels` 長度；`charts` 數＝佔位符數 |
| HTML 結構 | 兩版皆以 `</html>` 結尾；`mail_body` 無 `<svg>` / `<script>` |
| insight | LLM 回空值時仍無殘破片段；無 markdown、不以標題開頭、無「明年」字樣（皆警告） |
| 內容 | Major N 不少於上期；CHART1 當月標 (F)；品牌色與上期一致；缺填品牌 / 未填齊季度列出（皆警告，需人工確認後寄信） |

## 缺填品牌規則（v2）

- 品牌名單＝近 `brand_lookback_months` 個月內有送出 `yearly_bp_current` 的品牌（MNT 以 `APP_CONFIG.brand_order` 固定）。
- 名單內品牌當月未送出 → `on_missing_brand=warn`（預設）照常產出：1a / 5b 沿用該品牌最近一次填報並標 `＊`、Section 4 顯示「未填報」、
  季度與月度缺的月份以 `-` 呈現、Major 僅加總有數字的品牌；報告頂部紅色警示列出缺填品牌。`fail` 則該 app 本次不產出。
- 缺填品牌補填後重跑 job 即可覆蓋（MERGE 同列）。

## 注意事項

- 目標表 schema 變更（如加欄）屬一次性 migration，手動執行，不寫進本 job。
- 非增量：每次重產各 app 最新月，單一 MERGE 同列更新三欄，下游不會讀到新舊混合。


In [ ]:
%pip install -U anthropic httpx
# [c01] install
dbutils.library.restartPython()

In [ ]:
# [c02] source_check
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime

# ── 設定 ──
TARGET_TABLE = "micenter.mi3_datahub_prod.g_sales_brief_mail_report"
SRC_NUMERIC = "micenter.mi3_datahub_prod.g_sales_brief_numeric_entry"
SRC_BRAND_SUMMARY = "micenter.mi3_datahub_prod.g_sales_brief_brand_summary"
SRC_APP_SUMMARY = "micenter.mi3_datahub_prod.g_sales_brief_app_summary"
APP_CODES = ["TV", "NB", "MNT"]

# ── 參數（job parameters 可覆蓋）──
# on_missing_brand：品牌名單內有品牌當月未填時的處理
#   warn = 照常產出，缺填品牌沿用其最近一次填報並在報告標註（預設）
#   fail = 該 app 本次不產出（其他 app 照常）
# brand_lookback_months：品牌名單 = 近 N 個月內有送出年度 BP 的品牌（避免只看當月而漏掉晚填的品牌）
dbutils.widgets.text("on_missing_brand", "warn")
dbutils.widgets.text("brand_lookback_months", "6")
ON_MISSING_BRAND = dbutils.widgets.get("on_missing_brand").strip().lower()
BRAND_LOOKBACK_MONTHS = int(dbutils.widgets.get("brand_lookback_months"))
assert ON_MISSING_BRAND in ("warn", "fail"), f"on_missing_brand 必須是 warn / fail，收到 {ON_MISSING_BRAND!r}"

# ── 來源表最新月份（三表 period 交集，再取各 app 最新）──
source_latest = spark.sql(f"""
    WITH numeric_periods AS (
        SELECT DISTINCT app_code, fill_year, fill_month
        FROM {SRC_NUMERIC}
        WHERE status = 'submitted'
    ),
    brand_periods AS (
        SELECT DISTINCT app_code, fill_year, fill_month
        FROM {SRC_BRAND_SUMMARY}
    ),
    app_periods AS (
        SELECT DISTINCT app_code, fill_year, fill_month
        FROM {SRC_APP_SUMMARY}
    ),
    common AS (
        SELECT n.app_code, n.fill_year, n.fill_month
        FROM numeric_periods n
        INNER JOIN brand_periods b
            ON n.app_code = b.app_code AND n.fill_year = b.fill_year AND n.fill_month = b.fill_month
        INNER JOIN app_periods a
            ON n.app_code = a.app_code AND n.fill_year = a.fill_year AND n.fill_month = a.fill_month
    )
    SELECT app_code,
           MAX(fill_year * 100 + fill_month) AS source_latest_period
    FROM common
    GROUP BY app_code
""").collect()

source_map = {row["app_code"]: row["source_latest_period"] for row in source_latest}
print("來源表最新月份（三表交集）:", source_map)


def _period_minus(period, n):
    """yyyymm 整數往前推 n 個月。"""
    y, m = period // 100, period % 100
    idx = y * 12 + (m - 1) - n
    return (idx // 12) * 100 + idx % 12 + 1


# ── 品牌填報完整度：品牌名單（近 N 個月有送出年度 BP）vs 當月已送出 ──
# 回傳 {app: {"universe": [...brand_name], "filled": [...], "missing": [...]}}
completeness = {}
for app, period in source_map.items():
    lookback_from = _period_minus(period, BRAND_LOOKBACK_MONTHS - 1)
    rows = spark.sql(f"""
        SELECT brand_name,
               MAX(fill_year * 100 + fill_month) AS last_period
        FROM {SRC_NUMERIC}
        WHERE app_code = '{app}' AND status = 'submitted'
          AND category = 'panel_buy' AND metric = 'yearly_bp_current'
          AND target_year * 100 + target_month = fill_year * 100 + fill_month
          AND fill_year * 100 + fill_month BETWEEN {lookback_from} AND {period}
        GROUP BY brand_name
    """).collect()
    universe = sorted(r["brand_name"] for r in rows)
    filled = sorted(r["brand_name"] for r in rows if r["last_period"] == period)
    missing = [b for b in universe if b not in filled]
    completeness[app] = {"universe": universe, "filled": filled, "missing": missing}

# ── 決定本次產出清單 ──
apps_to_process = []
for app in APP_CODES:
    src = source_map.get(app)
    if not src:
        print(f"⏭️ {app}: 無來源數據 → 跳過")
        continue
    comp = completeness.get(app, {"universe": [], "filled": [], "missing": []})
    if comp["missing"]:
        msg = (f"{app} {src}: 品牌名單 {len(comp['universe'])} 個，當月未填 {comp['missing']}"
               f"（近 {BRAND_LOOKBACK_MONTHS} 個月有填報）")
        if ON_MISSING_BRAND == "fail":
            print(f"❌ {msg} → on_missing_brand=fail，本次不產出")
            continue
        print(f"⚠️ {msg} → 照常產出，缺填品牌沿用最近一次填報並標註")
    else:
        print(f"✅ {app} {src}: 品牌名單 {comp['universe']} 全數已填")
    # 缺填品牌的實際處理（沿用、標 ＊、季度 '-'）在 c03 stale_info / c05 做；這裡只決定產不產
    apps_to_process.append({
        "app_code": app,
        "fill_year": src // 100,
        "fill_month": src % 100,
    })

print(f"\n本次需產出: {[a['app_code'] for a in apps_to_process]}")


In [ ]:
# [c03] helpers
from collections import defaultdict
import calendar
import json
import re

# ═══ APP 特殊設定（不影響 TV/NB 通用性）═══
# brand_colors：品牌固定色（信件 PNG 與附件 SVG 共用）。未列的品牌依出現順序從 FALLBACK_COLORS 取色。
#   以 202609_2（已寄出版）的配色為準，之後排名變動不再換色。
APP_CONFIG = {
    "TV": {
        "display_name": "TV",
        "dashboard_url": "https://mi30.azurewebsites.net/shell/reportdashboard?groupId=102",
        # TV 燈號：≤ 黃燈門檻＝綠、> 黃燈門檻＝黃、≥ 紅燈門檻＝紅（黃燈為嚴格大於，與 MNT/NB 的 ≥ 不同）
        "inv_yellow_strict": True,
        "inv_thresholds": {
            "Samsung": (16, 14),
            "_default": (10, 7),   # Hisense / TCL / TPV
        },
        "y_prev_override": {
            "Samsung": 33.8,
            "Hisense": 30.0,
            "TCL": 30.0,
            "TPV": 6.2,
        },
        "brand_colors": {"TCL": "#2a78d6", "Samsung": "#eb6834", "Hisense": "#1baf7a", "TPV": "#eda100"},
    },
    "MNT": {
        "display_name": "Monitor",
        "dashboard_url": "https://mi30.azurewebsites.net/shell/reportdashboard?groupId=75",
        # brand_order：固定品牌名單與順序（不設則＝近 N 個月有填年度 BP 的品牌，依最新年度 BP 排序）
        "brand_order": ["Dell", "HP", "Samsung", "ASUS"],
        "brand_display": {"Asus": "ASUS"},  # DB raw → 顯示名稱
        "y25_yoy_override": {"Dell": -5, "HP": -23, "Samsung": -6, "ASUS": 27},
        # Major 合計列的 Y25 YoY%（y_prev_yoy）寫死值；不設或設 None＝維持以加總值計算
        "major_y25_yoy_override": -7,
        # inv_thresholds 格式：(red_high, yellow_high) 或 (red_high, yellow_high, red_low)
        #   2 元組＝只判斷上緣紅燈；3 元組＝低於 red_low 也是紅燈（過低／斷料風險）
        "inv_thresholds": {
            "ASUS": (24, 20),
            "_default": (20, 12),
        },
        "brand_colors": {"Dell": "#2a78d6", "HP": "#eb6834", "Samsung": "#1baf7a", "ASUS": "#eda100"},
    },
    "NB": {
        "display_name": "NB",
        "dashboard_url": "https://mi30.azurewebsites.net/shell/reportdashboard?groupId=95",
        # NB 庫存水位遠低於 MNT/TV：
        #   >= 10w 紅燈（偏高）／>= 5w 黃燈／3w~5w 綠燈／< 3w 紅燈（過低）
        "inv_thresholds": {
            "_default": (10, 5, 3),
        },
        "brand_colors": {"Lenovo": "#2a78d6", "HP": "#eb6834", "Dell": "#1baf7a",
                         "Asus": "#eda100", "Acer": "#9b59b6", "MSI": "#e74c3c"},
    }
}
FALLBACK_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9b59b6", "#e74c3c", "#16a085", "#7f8c8d"]
STALE_MARK = "＊"   # 表內標記：該品牌本月未填，數字沿用最近一次填報


def _normalize_brand(raw_brand_name, app_code):
    """從前綴品牌名萃取並正規化（如 MNT_Asus → ASUS）"""
    brand = raw_brand_name.split("_", 1)[1] if "_" in raw_brand_name else raw_brand_name
    return APP_CONFIG.get(app_code, {}).get("brand_display", {}).get(brand, brand)


def _period(fill_year, fill_month):
    return fill_year * 100 + fill_month


def _period_minus(period, n):
    """yyyymm 整數往前推 n 個月（c02 亦有同名定義，供 c02 先行使用）。"""
    y, m = period // 100, period % 100
    idx = y * 12 + (m - 1) - n
    return (idx // 12) * 100 + idx % 12 + 1


def _period_label(period):
    """yyyymm → Sep'26"""
    return f"{calendar.month_abbr[period % 100]}'{(period // 100) % 100}"


def brand_colors_for(app_code, brands_ordered):
    """回傳 {brand: color}；設定檔有的用固定色，其餘依序取 FALLBACK_COLORS 中尚未用到的色。"""
    fixed = APP_CONFIG.get(app_code, {}).get("brand_colors", {})
    used = set(fixed.get(b) for b in brands_ordered if b in fixed)
    spare = [c for c in FALLBACK_COLORS if c not in used]
    out = {}
    for b in brands_ordered:
        if b in fixed:
            out[b] = fixed[b]
        else:
            out[b] = spare.pop(0) if spare else FALLBACK_COLORS[len(out) % len(FALLBACK_COLORS)]
    return out


# ═══ 報告標題 / 系統連結（附件版與 mail_body 共用）═══
def report_title(app_code, fill_year, fill_month):
    """例：DSBG MI3.0 | Sep'26 Monitor 前線業務快報"""
    name = APP_CONFIG.get(app_code, {}).get("display_name", app_code)
    return f"DSBG MI3.0 | {calendar.month_abbr[fill_month]}'{fill_year % 100} {name} 前線業務快報"


def dashboard_link(app_code):
    """回傳 (url, 連結文字)；未設定 url 時回傳 (None, None)，呼叫端不放連結。"""
    cfg = APP_CONFIG.get(app_code, {})
    url = cfg.get("dashboard_url")
    if not url:
        return None, None
    return url, f"&#128279; 進入 MI3.0 系統查看 {cfg.get('display_name', app_code)} 報表 &#8594;"


# ═══ 庫存燈號判斷（附件版 Section 5b 與 mail_body Section 5b 共用）═══
def _get_inv_thresholds(app_code, brand):
    """回傳 (red_high, yellow_high, red_low)；red_low 為 None 代表不判斷過低。"""
    cfg = APP_CONFIG.get(app_code, {}).get("inv_thresholds", {})
    th = cfg.get(brand, cfg.get("_default", (20, 12)))
    red_high = th[0]
    yellow_high = th[1]
    red_low = th[2] if len(th) > 2 else None
    return red_high, yellow_high, red_low


def get_inv_lamp(app_code, brand, weeks):
    """依 app/brand 閾值回傳 (color, lamp_txt, desc)。
    判斷順序：偏高紅燈 → 過低紅燈 → 黃燈 → 綠燈。
    inv_yellow_strict（TV）：黃燈用 > yellow_high，其餘 app 用 >=。
    以原始週數（含小數）判斷，不做四捨五入。"""
    red_high, yellow_high, red_low = _get_inv_thresholds(app_code, brand)
    strict = APP_CONFIG.get(app_code, {}).get("inv_yellow_strict", False)
    if weeks >= red_high:
        return "#C00000", "紅燈", "庫存偏高"
    if red_low is not None and weeks < red_low:
        return "#C00000", "紅燈", "庫存過低，留意斷料風險"
    if (weeks > yellow_high) if strict else (weeks >= yellow_high):
        return "#BF8F00", "黃燈", "升高需觀察"
    return "#2E7D32", "綠燈", "庫存健康"


def get_inv_note(app_code, brands_ordered=None):
    """依 app 閾值產生 Section 5b 註解文字（HTML 用，'<' '>' 已跳脫）。"""
    cfg = APP_CONFIG.get(app_code, {}).get("inv_thresholds", {})
    if APP_CONFIG.get(app_code, {}).get("inv_yellow_strict"):
        ov = [b for b in cfg if b != "_default"]
        groups = [(b, cfg[b]) for b in ov]
        others = sorted(b for b in (brands_ordered or []) if b not in ov)
        if others and "_default" in cfg:
            groups.append((" / ".join(others), cfg["_default"]))
        rule = "；".join(
            f"{name}：≤ {th[1]}w 綠燈、&gt; {th[1]}w 黃燈、≥ {th[0]}w 紅燈" for name, th in groups
        )
        return f"註：庫存週數基於業務填報數據，燈號閾值依品牌分別設定——{rule}。"
    red_high, yellow_high, red_low = _get_inv_thresholds(app_code, "_default")
    if red_low is not None:
        rule = f"≥ {red_high}w 紅燈（偏高）、≥ {yellow_high}w 黃燈、{red_low}w ~ {yellow_high}w 綠燈、&lt; {red_low}w 紅燈（過低）"
    else:
        rule = f"≥ {red_high}w 紅燈、≥ {yellow_high}w 黃燈、&lt; {yellow_high}w 綠燈"
    note = f"註：庫存週數基於業務填報數據，燈號閾值：{rule}。"
    ov = [b for b in cfg.keys() if b != "_default"]
    if ov:
        detail = "；".join(
            f"{b} ≥{cfg[b][0]}w 紅燈／≥{cfg[b][1]}w 黃燈" for b in ov
        )
        note += f"（品牌例外：{detail}）"
    return note


# ═══ 缺填品牌（stale）共用 ═══
def stale_info(brands_ordered, yearly_data, fill_year, fill_month):
    """回傳 {brand: as_of_period 或 None}，只含本月未填的品牌。
    as_of = 該品牌 y_cur / inv_weeks 實際取自哪一期；None＝回溯期內完全沒填。"""
    cur = _period(fill_year, fill_month)
    out = {}
    for b in brands_ordered:
        as_of = yearly_data.get(b, {}).get("as_of")
        if as_of is None or as_of < cur:
            out[b] = as_of
    return out


def brand_label(brand, stale):
    return f"{brand}{STALE_MARK}" if brand in stale else brand


def stale_note_text(stale, fill_year, fill_month):
    """表格註解用：＊Samsung 本月未填，沿用 Sep'26 填報。無缺填回傳空字串。"""
    if not stale:
        return ""
    parts = []
    for b, as_of in stale.items():
        parts.append(f"{b} 沿用 {_period_label(as_of)} 填報" if as_of else f"{b} 近期無填報")
    return f"{STALE_MARK}{_period_label(_period(fill_year, fill_month))} 未填：" + "；".join(parts) + "。"


def missing_banner_text(stale, partial_quarters, fill_year, fill_month):
    """報告頂部警示文字（html 與 mail 共用）；無缺填回傳空字串。"""
    if not stale:
        return ""
    cur_label = _period_label(_period(fill_year, fill_month))
    txt = f"本期 {cur_label} 尚未填報：{'、'.join(stale.keys())}。表內標 {STALE_MARK} 者沿用該品牌最近一次填報"
    txt += "；月份未填齊的季度與月度欄位以「-」表示，Major 合計僅加總有數字的品牌。"
    return txt


# ═══ Major 合計列（Section 1a / 1b 的第一列，附件版與 mail 版共用）═══
def _sum_opt(values):
    """加總並忽略 None；全部為 None 時回傳 None。"""
    vals = [v for v in values if v is not None]
    return sum(vals) if vals else None


def build_major_yearly(brands_ordered, yearly_data, app_code=""):
    """回傳 Major 合計列，結構與 yearly_data 的單一品牌 dict 相同。
    各金額欄為所有品牌加總；YoY% / Hit% 一律以加總後的值重算。
    例外：APP_CONFIG[app_code].major_y25_yoy_override 直接覆蓋 Y25 YoY%。"""
    def g(key):
        return _sum_opt([yearly_data.get(b, {}).get(key) for b in brands_ordered])

    y_prev2 = g("y_prev2")
    y_prev = g("y_prev")
    y_pre = g("y_cur_pre")
    y_cur = g("y_cur")
    ytd = g("ytd")

    def _yoy(cur, base):
        return ((cur - base) / base * 100) if (cur is not None and base) else None

    row = {
        "y_prev2": y_prev2,
        "y_prev": y_prev,
        "y_cur_pre": y_pre,
        "y_cur": y_cur,
        "y_prev_yoy": _yoy(y_prev, y_prev2),
        "y_cur_pre_yoy": _yoy(y_pre, y_prev),
        "y_cur_yoy": _yoy(y_cur, y_prev),
        "ytd": ytd,
        "hit_pct": (ytd / y_cur * 100) if (ytd is not None and y_cur) else None,
        "inv_weeks": None,
    }
    major_override = APP_CONFIG.get(app_code, {}).get("major_y25_yoy_override")
    if major_override is not None:
        row["y_prev_yoy"] = major_override
    return row


def build_major_quarterly(brands_ordered, quarterly_data, quarters):
    """回傳 {(year, quarter): 合計值}；只加總該季有完整數字的品牌（未填齊的品牌已在 get_quarterly_data 轉 None）。"""
    return {
        key: _sum_opt([quarterly_data.get(b, {}).get(key) for b in brands_ordered])
        for key in quarters
    }


def build_major_mom(brands_ordered, monthly_series, periods, base_period):
    """Major 列 MoM%：每個月只拿「前後兩月都有數字」的品牌做同口徑加總，
    避免某品牌缺最後一個月時 Major 合計驟降（D: 202610 MNT Dec -52% 失真）。
    回傳 {period: pct 或 None}。"""
    by_brand = {b: {(e["year"], e["month"]): e["value"] for e in monthly_series.get(b, [])}
                for b in brands_ordered}
    out = {}
    prev_p = base_period
    for p in periods:
        if prev_p is None:
            out[p] = None
        else:
            cur_sum = prev_sum = 0.0
            n = 0
            for b in brands_ordered:
                cv, pv = by_brand[b].get(p), by_brand[b].get(prev_p)
                if cv is not None and pv is not None:
                    cur_sum += cv
                    prev_sum += pv
                    n += 1
            out[p] = ((cur_sum - prev_sum) / prev_sum * 100) if (n and prev_sum) else None
        prev_p = p
    return out


def major_label(brands_ordered):
    """Major 列的顯示名稱，例如 6 個品牌 → 'Major 6'。"""
    return f"Major {len(brands_ordered)}"


# ═══ LLM Helper（尚未初始化時用 fallback）═══
_LLM_READY = True

def call_claude(messages, max_tokens=2048, thinking=False):
    """Placeholder - will be overridden by LLM init cell."""
    return ""


_INSIGHT_HEAD_RE = re.compile(
    r"^(?:(?:高階主管|趨勢|月度|季度|年度)?(?:採購)?(?:趨勢)?(?:（分析）|\(分析\))?(?:判讀|分析|摘要)[:：]\s*)+"
)

def clean_insight(text):
    """LLM 輸出後處理：去 markdown 記號、去開頭重複標題、收斂空白。
    D: 202610 MNT 季度分析出現 **…**；月度判讀開頭「月度採購趨勢摘要：」與固定標題重複。"""
    if not text:
        return text
    t = text.strip()
    m = re.match(r"^\*\*([^*\n]{1,30})\*\*\s*", t)   # 開頭一整段粗體且短、無句點 → 視為標題整段去掉
    if m and "。" not in m.group(1):
        t = t[m.end():]
    t = re.sub(r"\*\*(.+?)\*\*", r"\1", t)          # **bold**
    t = re.sub(r"(?m)^\s*#{1,6}\s*", "", t)          # # heading
    t = re.sub(r"(?m)^\s*[-*•]\s+", "", t)           # - bullet
    t = t.replace("**", "").replace("__", "")
    t = re.sub(r"\s*\n+\s*", " ", t)                 # 單段輸出
    t = _INSIGHT_HEAD_RE.sub("", t).strip()
    t = re.sub(r"^[「『\"']?(?:摘要|分析)[:：]\s*", "", t).strip()
    return t


def generate_insight(section_name, data_summary):
    """Call LLM to generate insight paragraph. Falls back to empty if LLM not ready."""
    if not _LLM_READY:
        return "（AI 分析尚未啟用）"
    prompt = f"""你是 AUO 顯示器業務分析師。請根據以下數據，用繁體中文撰寫一段中性、簡潔的高階主管摘要（100-200字）。
要求：點出各品牌主要趨勢、變化原因、風險提示，本區重點是資訊正確的解讀，不要用過多解釋。
格式：純文字一段，不要 markdown 符號（不要 **、#、條列），不要加標題、不要以「摘要：」「分析：」開頭，直接從第一句分析開始。
數據中標 (F) 為預估值、標「沿用」為該品牌本月未填而沿用上期的值、「-」為未填，請勿把「-」或沿用值解讀為衰退。

Section: {section_name}
數據:
{data_summary}"""
    raw = call_claude([{"role": "user", "content": prompt}], max_tokens=500, thinking=False)
    return clean_insight(raw) or "（AI 分析產生失敗）"


# ═══════════ 品牌名單與排序 ═══════════
def get_brand_order(app_code, fill_year, fill_month, lookback_months=6):
    """品牌名單（顯示名）。
    APP_CONFIG.brand_order 有設 → 固定名單與順序。
    否則 → 近 lookback_months 個月內有送出年度 BP 的品牌，依各自最新一筆年度 BP 由大到小排序。
    （v1 只抓當月送出的品牌，晚填的品牌會整個消失：202610 NB 少了 Dell / Acer。）"""
    if app_code in APP_CONFIG and "brand_order" in APP_CONFIG[app_code]:
        return list(APP_CONFIG[app_code]["brand_order"])
    period = _period(fill_year, fill_month)
    lookback_from = _period_minus(period, lookback_months - 1)
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'yearly_bp_current'
              AND target_year * 100 + target_month = fill_year * 100 + fill_month
              AND fill_year * 100 + fill_month BETWEEN {lookback_from} AND {period}
        )
        SELECT brand_code, brand_name, value AS bp FROM ranked WHERE rn = 1
        ORDER BY bp DESC NULLS LAST, brand_name
    """).collect()
    return [_normalize_brand(r["brand_name"], app_code) for r in rows]


# ═══════════ 年度數據 (Section 1a) ═══════════
def get_yearly_data(app_code, fill_year, fill_month, lookback_months=6):
    """Get Y24, Y25, Y26(Pre), Y26(Cur), YTD, inv_weeks per brand for Section 1a.
    每個品牌多回傳 as_of（y_cur / inv_weeks 取自哪一期 yyyymm）；as_of < 本期 代表本月未填、沿用上期。
    - Cur：各品牌「最近一次」送出的年度 BP（target 期＝填報期），回溯 lookback_months。
    - Pre：該品牌 as_of 之前一次送出的年度 BP（只看 target 期＝填報期的列，
           排除之後補送 / 重送 target 為舊月份的資料 → Q8）。
    - Y_prev / Y_prev2：前一 / 前兩年 target_month=12 的最新值。
    - YTD：target_month 1~(fill_month-1) 各月最新值加總（不含當月，當月尚未結算）。"""
    period = _period(fill_year, fill_month)
    lookback_from = _period_minus(period, lookback_months - 1)

    # Cur + inv_weeks：各品牌最近一期（每期內以最新 submission 為準）
    cur = spark.sql(f"""
        WITH per_sub AS (
            SELECT brand_code, brand_name,
                   fill_year * 100 + fill_month AS p,
                   MAX(CASE WHEN category='panel_buy' AND metric='yearly_bp_current' THEN value END) AS y_cur,
                   MAX(CASE WHEN category='inventory' AND metric='monthly' THEN value END) AS inv_weeks,
                   MAX(submitted_at) AS submitted_at, MAX(submission_id) AS submission_id
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND target_year * 100 + target_month = fill_year * 100 + fill_month
              AND fill_year * 100 + fill_month BETWEEN {lookback_from} AND {period}
            GROUP BY brand_code, brand_name, fill_year, fill_month, submission_id
        ),
        ranked AS (
            SELECT *, ROW_NUMBER() OVER (
                       PARTITION BY brand_code
                       ORDER BY p DESC, submitted_at DESC, submission_id DESC) AS rn
            FROM per_sub
            WHERE y_cur IS NOT NULL
        )
        SELECT brand_code, brand_name, p AS as_of, y_cur, inv_weeks FROM ranked WHERE rn = 1
    """).collect()

    # Pre：每個品牌各期的年度 BP（最新 submission），後面用 python 取 as_of 之前的一期
    hist = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, fill_year * 100 + fill_month AS p, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, fill_year, fill_month
                       ORDER BY submitted_at DESC, submission_id DESC) AS rn
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'yearly_bp_current'
              AND target_year * 100 + target_month = fill_year * 100 + fill_month
              AND fill_year * 100 + fill_month <= {period}
        )
        SELECT brand_code, p, value FROM ranked WHERE rn = 1
    """).collect()
    hist_map = defaultdict(dict)   # brand_code -> {p: value}
    for r in hist:
        hist_map[r["brand_code"]][r["p"]] = r["value"]

    def _latest_year_end(target_year):
        rows = spark.sql(f"""
            WITH ranked AS (
                SELECT brand_code, value,
                       ROW_NUMBER() OVER (
                           PARTITION BY brand_code
                           ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                       ) AS rn
                FROM {SRC_NUMERIC}
                WHERE app_code = '{app_code}' AND target_year = {target_year} AND target_month = 12
                  AND category = 'panel_buy' AND metric = 'yearly_bp_current'
                  AND status = 'submitted'
            )
            SELECT brand_code, value FROM ranked WHERE rn = 1
        """).collect()
        return {r["brand_code"]: float(r["value"]) for r in rows if r["value"] is not None}

    y_prev_map = _latest_year_end(fill_year - 1)
    y_prev2_map = _latest_year_end(fill_year - 2)

    ytd_rows = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, target_month, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'monthly'
              AND target_year = {fill_year} AND target_month < {fill_month}
        )
        SELECT brand_code, SUM(value) AS ytd
        FROM ranked WHERE rn = 1
        GROUP BY brand_code
    """).collect()
    ytd_map = {r["brand_code"]: float(r["ytd"]) if r["ytd"] is not None else None for r in ytd_rows}

    def _f(v):
        return float(v) if v is not None else None

    result = {}
    for r in cur:
        bc = r["brand_code"]
        brand = _normalize_brand(r["brand_name"], app_code)
        as_of = int(r["as_of"])
        y26c = _f(r["y_cur"])
        earlier = [p for p in hist_map.get(bc, {}) if p < as_of]
        y26p = _f(hist_map[bc][max(earlier)]) if earlier else None
        y_pv = y_prev_map.get(bc)
        y_p2v = y_prev2_map.get(bc)
        ytd = ytd_map.get(bc)
        result[brand] = {
            "as_of": as_of,
            "y_prev2": y_p2v, "y_prev": y_pv, "y_cur_pre": y26p, "y_cur": y26c,
            "y_prev_yoy": ((y_pv - y_p2v) / y_p2v * 100) if (y_pv and y_p2v) else None,
            "y_cur_pre_yoy": ((y26p - y_pv) / y_pv * 100) if (y26p and y_pv) else None,
            "y_cur_yoy": ((y26c - y_pv) / y_pv * 100) if (y26c and y_pv) else None,
            "ytd": ytd,
            "hit_pct": (ytd / y26c * 100) if (ytd and y26c and y26c > 0) else None,
            "inv_weeks": _f(r["inv_weeks"]),
        }

    # y_prev baseline override（分析師提供的前年 baseline，覆蓋 DB 查詢值並重算相關 YoY）
    if app_code in APP_CONFIG and "y_prev_override" in APP_CONFIG[app_code]:
        overrides = APP_CONFIG[app_code]["y_prev_override"]
        for brand in result:
            if brand in overrides:
                y_pv = overrides[brand]
                result[brand]["y_prev"] = y_pv
                y_p2v = result[brand]["y_prev2"]
                y26p = result[brand]["y_cur_pre"]
                y26c = result[brand]["y_cur"]
                result[brand]["y_prev_yoy"] = ((y_pv - y_p2v) / y_p2v * 100) if (y_pv and y_p2v) else None
                result[brand]["y_cur_pre_yoy"] = ((y26p - y_pv) / y_pv * 100) if (y26p and y_pv) else None
                result[brand]["y_cur_yoy"] = ((y26c - y_pv) / y_pv * 100) if (y26c and y_pv) else None

    # Y_prev YoY override（分析師手動提供 YoY 數字，直接覆寫計算值）
    if app_code in APP_CONFIG and "y25_yoy_override" in APP_CONFIG[app_code]:
        overrides = APP_CONFIG[app_code]["y25_yoy_override"]
        for brand in result:
            if brand in overrides:
                result[brand]["y_prev_yoy"] = overrides[brand]
    return result


# ═══════════ 季度數據 (Section 1b) ═══════════
def get_quarterly_data(app_code, fill_year, fill_month):
    """回傳 (quarterly, partial)：
    quarterly = {brand: {(year, q): 三個月加總}}，只含三個月都有數字的季度；
    partial   = {brand: {(year, q): 已有的月份數}}，月份未填齊的季度（顯示為 '-'，並在註解列出）。
    v1 把只加到部分月份的季度當完整值顯示（202610 MNT Samsung 26 4Q=1.4 → QoQ -42% 失真）。"""
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT target_year, target_month, brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_year, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'monthly'
              AND target_year >= {fill_year - 1}
        )
        SELECT target_year, target_month, brand_code, brand_name, value
        FROM ranked WHERE rn = 1
    """).collect()

    sums = defaultdict(lambda: defaultdict(float))
    months = defaultdict(lambda: defaultdict(set))
    for r in rows:
        if r["value"] is None:
            continue
        bn = _normalize_brand(r["brand_name"], app_code)
        key = (r["target_year"], (r["target_month"] - 1) // 3 + 1)
        sums[bn][key] += float(r["value"])
        months[bn][key].add(r["target_month"])

    quarterly, partial = {}, {}
    for bn in sums:
        quarterly[bn] = {}
        for key, total in sums[bn].items():
            if len(months[bn][key]) == 3:
                quarterly[bn][key] = total
            else:
                partial.setdefault(bn, {})[key] = len(months[bn][key])
    return quarterly, partial


# ═══════════ 月度趨勢 (Section 2a SVG + 2b MoM) ═══════════
def get_monthly_series(app_code, fill_year, fill_month, n_months=18):
    """Return monthly panel_buy for SVG chart & MoM table.
    使用 target_year/target_month 作為時間軸，每個 target_month 取最新提交值。
    包含當月 submission 的未來預測月份（供 MoM 表使用）。"""
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT target_year, target_month, brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_year, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'monthly'
        )
        SELECT target_year, target_month, brand_code, brand_name, value
        FROM ranked WHERE rn = 1
        ORDER BY target_year, target_month
    """).collect()

    brand_series = defaultdict(list)
    for r in rows:
        bn = _normalize_brand(r["brand_name"], app_code)
        brand_series[bn].append({
            "year": r["target_year"], "month": r["target_month"],
            "value": float(r["value"]) if r["value"] is not None else None
        })
    for brand in brand_series:
        brand_series[brand] = brand_series[brand][-n_months:]
    return dict(brand_series)


# ═══════════ 庫存週數歷史趨勢 (Section 5a SVG) ═══════════
def get_inventory_series(app_code, fill_year, fill_month, n_months=13):
    """Return monthly inventory weeks per brand (last n_months).
    使用 target_year/target_month 作為時間軸，每個 target_month 取最新提交值。"""
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT target_year, target_month, brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_year, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM {SRC_NUMERIC}
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'inventory' AND metric = 'monthly'
              AND (target_year * 100 + target_month) <= {fill_year * 100 + fill_month}
        )
        SELECT target_year, target_month, brand_code, brand_name, value
        FROM ranked WHERE rn = 1
        ORDER BY target_year, target_month
    """).collect()

    brand_series = defaultdict(list)
    for r in rows:
        bn = _normalize_brand(r["brand_name"], app_code)
        if r["value"] is not None:
            brand_series[bn].append({
                "year": r["target_year"], "month": r["target_month"],
                "value": float(r["value"])
            })
    for brand in brand_series:
        brand_series[brand] = brand_series[brand][-n_months:]
    return dict(brand_series)


# ═══════════ 摘要數據 (Section 3 & 4) ═══════════
def get_summary_data(app_code, fill_year, fill_month):
    """Get app-level and brand-level AI summaries."""
    app_row = spark.sql(f"""
        SELECT * FROM {SRC_APP_SUMMARY}
        WHERE app_code = '{app_code}' AND fill_year = {fill_year} AND fill_month = {fill_month}
    """).collect()
    brand_rows = spark.sql(f"""
        SELECT * FROM {SRC_BRAND_SUMMARY}
        WHERE app_code = '{app_code}' AND fill_year = {fill_year} AND fill_month = {fill_month}
    """).collect()
    return app_row[0].asDict() if app_row else None, brand_rows


# ═══════════ 圖表中間結構（供 SVG 與 chart_data JSON 同源共用）═══════════
def extract_chart_intermediate(brands_ordered, monthly_series, inv_series, fill_year, fill_month, app_code=""):
    """Extract chart data as intermediate dict shared by SVG (2a / 5a) and chart_data JSON.
    - 預估段：當月（含）之後為 (F)。當月為本期填報的預估、尚未結算（與 YTD「不含當月」同一定義；
      202609_2 寄出前手動把 Sep 改成 Sep(F) 即是此規則）。
    - 顏色：依品牌固定（brand_colors_for），不再依排名位置取色。
    Returns dict with 'chart1' (monthly panel buy) and 'chart2' (inventory weeks)."""
    colors = brand_colors_for(app_code, brands_ordered)
    cur_period = _period(fill_year, fill_month)

    # ── Chart 1: 品牌月度面板採購趨勢 ──
    all_periods = set()
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    periods1 = sorted(all_periods)[-15:]  # last 15 months (same as SVG)

    x_labels1 = []
    forecast_from_index = None
    for i, (yr, mo) in enumerate(periods1):
        is_forecast = (yr * 100 + mo) >= cur_period
        if is_forecast and forecast_from_index is None:
            forecast_from_index = i
        mon_abbr = calendar.month_abbr[mo]
        label = f"{yr % 100}-{mon_abbr}" if (i == 0 or yr != periods1[i - 1][0]) else mon_abbr
        if is_forecast:
            label += "(F)"
        x_labels1.append(label)

    def _series(src, periods, brands):
        out = []
        for brand in brands:
            period_val = {(e["year"], e["month"]): e["value"] for e in src.get(brand, [])}
            out.append({"name": brand, "color": colors[brand], "data": [period_val.get(p) for p in periods]})
        return out

    chart1 = {
        "periods": periods1,
        "x_labels": x_labels1,
        "forecast_from_index": forecast_from_index,
        "series": _series(monthly_series, periods1, brands_ordered[:4]),   # 信件 PNG 只放前 4 名（附件 SVG 放 6 個，使用者已確認）
        "series_svg": _series(monthly_series, periods1, brands_ordered[:6]),
    }

    # ── Chart 2: 主要品牌庫存週數趨勢 ──
    all_periods2 = set()
    for brand in brands_ordered:
        for entry in inv_series.get(brand, []):
            all_periods2.add((entry["year"], entry["month"]))
    periods2 = sorted(all_periods2)[-13:]  # last 13 months

    x_labels2 = []
    for i, (yr, mo) in enumerate(periods2):
        x_labels2.append(f"{yr % 100}-{mo:02d}" if (i == 0 or yr != periods2[i - 1][0]) else f"{mo:02d}")

    chart2 = {
        "periods": periods2,
        "x_labels": x_labels2,
        "forecast_from_index": None,  # inventory has no forecast segment
        "series": _series(inv_series, periods2, brands_ordered[:4]),
        "series_svg": _series(inv_series, periods2, brands_ordered[:6]),
    }
    return {"chart1": chart1, "chart2": chart2}


# ═══════════ 內容檢查（寫入前，對記憶體資料做；回傳警告清單）═══════════
def content_checks(ctx, prev_major_n=None, yearly_tolerance_pct=5.0):
    """回傳 [str]：品牌數比上期少、缺填品牌、季度未填齊、圖表尾端 null、年度 vs 月度加總差異、AI 文字殘留 markdown。
    只警告不擋；要擋的情境由 c02 on_missing_brand 控制。"""
    warns = []
    brands = ctx["brands_ordered"]
    fy = ctx["fill_year"]
    if prev_major_n is not None and len(brands) < prev_major_n:
        warns.append(f"品牌數 {len(brands)} 少於上期 Major {prev_major_n}")
    for b, as_of in ctx["stale"].items():
        warns.append(f"{b} 本月未填，沿用 {_period_label(as_of) if as_of else '（無）'}")
    for b, qs in ctx["partial_quarters"].items():
        for (y, q), n in sorted(qs.items()):
            if y >= fy - 1:
                warns.append(f"{b} {y % 100} {q}Q 只有 {n} 個月 → 顯示 '-'")
    for cid in ("chart1", "chart2"):
        for s in ctx["chart"][cid]["series"]:
            if s["data"] and s["data"][-1] is None:
                warns.append(f"{cid} {s['name']} 最後一點為 null")
    # 年度 BP vs 當年 12 個月加總（都有時才比）
    by_brand = {b: {(e["year"], e["month"]): e["value"] for e in ctx["monthly_series"].get(b, [])} for b in brands}
    for b in brands:
        y_cur = ctx["yearly_data"].get(b, {}).get("y_cur")
        vals = [by_brand[b].get((fy, m)) for m in range(1, 13)]
        if y_cur and all(v is not None for v in vals):
            s = sum(vals)
            diff = (s - y_cur) / y_cur * 100
            if abs(diff) > yearly_tolerance_pct:
                warns.append(f"{b} 年度 BP {y_cur:.1f} vs {fy} 月度加總 {s:.1f}（{diff:+.0f}%）")
    for k, txt in ctx["insights"].items():
        if re.search(r"(?m)\*\*|^#|\n- ", txt or ""):
            warns.append(f"insight {k} 殘留 markdown")
        if re.match(r"^(?:摘要|分析|趨勢判讀)[:：]", txt or ""):
            warns.append(f"insight {k} 以標題開頭")
    return warns


print("✅ 資料整理函式已定義")


In [ ]:
# [c04] llm_init
import httpx
import time as _time

FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
FOUNDRY_BASE_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
MODEL_SONNET = "claude-sonnet-5"

def call_claude(messages, max_tokens=2048, thinking=False):
    """Call Claude via Azure Foundry with retry."""
    for attempt in range(3):
        try:
            body = {"model": MODEL_SONNET, "max_tokens": max_tokens, "messages": messages}
            if not thinking:
                body["thinking"] = {"type": "disabled"}
            resp = httpx.post(
                FOUNDRY_BASE_URL,
                headers={"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"},
                json=body, timeout=120.0,
            )
            resp.raise_for_status()
            data = resp.json()
            text_parts = [block["text"] for block in data.get("content", []) if block.get("type") == "text"]
            return "\n".join(text_parts) if text_parts else ""
        except Exception as e:
            if attempt < 2:
                _time.sleep(10 * (attempt + 1))
            else:
                print(f"  \u26a0\ufe0f LLM call failed: {e}")
                return ""

globals()['_LLM_READY'] = True
print("\u2705 LLM \u5df2\u521d\u59cb\u5316\uff0cAI Insight \u5df2\u555f\u7528")

In [ ]:
# [c05] render
# ════════════════════════════════════════════════════════════
# Step 3: HTML 產出函式（完整 web 版，含 SVG 圖表 + AI Insight）
# 所有 render 函式吃同一個 ctx（build_report_context 在 c06 組好），不再各自取數。
# ════════════════════════════════════════════════════════════

def _fmt_num(v, decimals=1):
    if v is None: return "-"
    return f"{v:.{decimals}f}"

def _fmt_weeks(v):
    """庫存週數顯示，保留一位小數（燈號以原始值判斷）。"""
    return f"{v:.1f}w"

def _fmt_pct(v):
    if v is None: return "-"
    return f"{v:+.0f}%" if v != 0 else "0%"

def _tag_pct(v):
    """Return tag-styled percentage HTML."""
    if v is None: return "-"
    txt = f"{v:+.0f}%" if v != 0 else "0%"
    if v > 0:
        return f'<span class="tag-pos">{txt}</span>'
    elif v < 0:
        return f'<span class="tag-neg">{txt}</span>'
    return f'<span class="tag-neu">{txt}</span>'

def _month_label(year, month):
    return f"{year % 100}-{calendar.month_abbr[month]}"

def _q_label(yr, q):
    return f"{yr % 100} {q}Q"

def _partial_note(partial_quarters, brands_ordered, quarters):
    """1b 註解：列出顯示為 '-' 的季度（只列表內有顯示的季度）。"""
    items = []
    for b in brands_ordered:
        for key, n in sorted(partial_quarters.get(b, {}).items()):
            if key in quarters:
                items.append(f"{b} {_q_label(*key)} 僅 {n} 個月")
    return ("未填齊季度以「-」表示、不計入 Major：" + "；".join(items) + "。") if items else ""


# ═══ CSS ═══
HTML_CSS = """
<style>
*{box-sizing:border-box;margin:0;padding:0;}
body{font-family:'Segoe UI','Noto Sans TC','Microsoft JhengHei',Arial,sans-serif;background:#F3F4F6;color:#1A1A2E;font-size:14px;}
.container{max-width:960px;margin:0 auto;padding:16px;}
.header{background:linear-gradient(135deg,#1F4E79,#2E75B6);color:#fff;border-radius:12px;padding:26px 30px;margin-bottom:20px;}
.header h1{font-size:25px;font-weight:700;margin-bottom:8px;}
.header-meta{display:flex;flex-wrap:wrap;gap:12px;font-size:13px;opacity:0.92;}
.header-meta span{background:rgba(255,255,255,0.15);padding:4px 12px;border-radius:20px;}
.header-link{margin-top:14px;font-size:14px;}
.header-link a{color:#fff;font-weight:700;text-decoration:underline;}
.section-card{background:#fff;border-radius:12px;border:1px solid #C8D8E8;padding:22px 24px;margin-bottom:20px;}
.section-title{display:flex;align-items:center;gap:12px;margin-bottom:16px;}
.section-badge{width:28px;height:28px;border-radius:50%;background:#1F4E79;color:#fff;font-size:13px;font-weight:700;display:flex;align-items:center;justify-content:center;flex-shrink:0;}
.section-title h2{font-size:17px;font-weight:700;color:#1A1A2E;}
.sub-title{font-size:14px;font-weight:700;color:#1F4E79;margin:18px 0 10px;}
.chart-wrap{position:relative;padding-bottom:56%;height:0;overflow:hidden;border-radius:8px;border:1px solid #E0E0E0;background:#FCFCFB;}
.insight-box{background:#F5F7FA;border-left:4px solid #1F4E79;padding:12px 16px;border-radius:0 6px 6px 0;margin-top:14px;font-size:13px;line-height:1.8;color:#1A1A2E;}
.warn-box{background:#FDECEC;border:1px solid #F5B5B5;border-radius:8px;padding:12px 18px;margin-bottom:18px;font-size:13px;color:#8B1A1A;line-height:1.7;}
table{width:100%;border-collapse:collapse;font-size:13px;}
th{background:#1F4E79;color:#fff;padding:8px 9px;text-align:center;font-weight:600;white-space:nowrap;}
td{padding:7px 9px;border-bottom:1px solid #DCE6F0;}
tr:nth-child(even) td{background:#EFF5FB;}
tr:nth-child(odd) td{background:#fff;}
.td-num{text-align:right;font-variant-numeric:tabular-nums;}
.td-txt{text-align:left;}
.tbl-scroll{overflow-x:auto;}
.tag-pos{display:inline-block;background:#E8F5E9;color:#2E7D32;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neg{display:inline-block;background:#FDECEC;color:#C00000;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neu{display:inline-block;background:#FFF8E1;color:#946C00;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.lamp{display:inline-block;width:11px;height:11px;border-radius:50%;vertical-align:middle;margin-right:6px;}
.note{font-size:11px;color:#8A8A96;margin-top:8px;line-height:1.6;}
.footer{text-align:center;font-size:12px;color:#9E9E9E;padding:14px 0 24px;line-height:1.7;}
.facet-label{width:84px;background:#EDF2F7 !important;color:#1F4E79;font-weight:700;vertical-align:top;}
</style>
"""


# ═══ Section 1a: 年度 Panel Buy 總表 ═══
def _build_section1a(ctx):
    brands_ordered, yearly_data, stale = ctx["brands_ordered"], ctx["yearly_data"], ctx["stale"]
    fill_year, fill_month, app_code = ctx["fill_year"], ctx["fill_month"], ctx["app_code"]
    major_d = build_major_yearly(brands_ordered, yearly_data, app_code)
    show_prev_yoy = any(d.get('y_prev_yoy') is not None
                        for d in [major_d] + [yearly_data.get(b, {}) for b in brands_ordered])

    def _row(label, d, is_major=False):
        hit = d.get("hit_pct")
        bg = ' style="background:#EDF2F7 !important;"' if is_major else ''
        prev_yoy_td = (f'            <td class="td-num"{bg}>{_tag_pct(d.get("y_prev_yoy"))}</td>\n'
                       if show_prev_yoy else "")
        return f"""          <tr><td class="td-txt"{bg}><strong>{label}</strong></td>
            <td class="td-num"{bg}>{_fmt_num(d.get('y_prev'))}</td><td class="td-num"{bg}>{_fmt_num(d.get('y_cur_pre'))}</td>
            <td class="td-num"{bg}><strong>{_fmt_num(d.get('y_cur'))}</strong></td>
{prev_yoy_td}            <td class="td-num"{bg}>{_tag_pct(d.get('y_cur_pre_yoy'))}</td>
            <td class="td-num"{bg}>{_tag_pct(d.get('y_cur_yoy'))}</td>
            <td class="td-num"{bg}>{_fmt_num(d.get('ytd'))}</td>
            <td class="td-num"{bg}>{f'{hit:.0f}%' if hit is not None else '-'}</td></tr>\n"""

    rows_html = _row(major_label(brands_ordered), major_d, is_major=True)
    for brand in brands_ordered:
        rows_html += _row(brand_label(brand, stale), yearly_data.get(brand, {}))
    note = stale_note_text(stale, fill_year, fill_month)
    note_html = f'\n    <div class="note">{note}</div>' if note else ""

    return f"""    <div class="sub-title">1a｜年度 Panel Buy 總表（Unit: M pcs）</div>
    <div class="tbl-scroll"><table>
      <thead><tr><th>Brand</th><th>Y25</th><th>Y26 (Pre)</th><th>Y26 (Cur)</th>{"<th>Y25 YoY%</th>" if show_prev_yoy else ""}<th>Y26 (Pre) YoY%</th><th>Y26 YoY%</th><th>YTD (1~{fill_month - 1}M)</th><th>Hit%</th></tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>{note_html}"""


# ═══ Section 1b: 季度 Panel Buy 明細 ═══
def _quarters_for(fill_year, fill_month):
    cur_q = (fill_month - 1) // 3 + 1
    return cur_q, [(fill_year - 1, q) for q in range(1, 5)] + [(fill_year, q) for q in range(1, cur_q + 1)]


def _build_section1b(ctx):
    brands_ordered, quarterly_data, stale = ctx["brands_ordered"], ctx["quarterly_data"], ctx["stale"]
    fill_year, fill_month = ctx["fill_year"], ctx["fill_month"]
    cur_q, quarters = _quarters_for(fill_year, fill_month)

    header = "<th>Brand</th>"
    for yr, q in quarters:
        label = _q_label(yr, q) + (" (Cur)" if (yr == fill_year and q == cur_q) else "")
        header += f"<th>{label}</th>"
    for q in range(1, cur_q + 1):
        header += f"<th>{fill_year % 100} {q}Q QoQ</th>"

    def _row(label, bq, is_major=False):
        bg = ' style="background:#EDF2F7 !important;"' if is_major else ''
        row = f'<td class="td-txt"{bg}><strong>{label}</strong></td>'
        qvals = []
        for yr, q in quarters:
            v = bq.get((yr, q))
            qvals.append(v)
            if yr == fill_year and q == cur_q:
                row += f'<td class="td-num"{bg}><strong>{_fmt_num(v)}</strong></td>'
            else:
                row += f'<td class="td-num"{bg}>{_fmt_num(v)}</td>'
        for qi in range(len(quarters) - cur_q, len(quarters)):
            cur_v = qvals[qi]
            prev_v = qvals[qi - 1] if qi > 0 else None
            qoq = ((cur_v - prev_v) / prev_v * 100) if (cur_v and prev_v and prev_v != 0) else None
            row += f'<td class="td-num"{bg}>{_tag_pct(qoq)}</td>'
        return f"          <tr>{row}</tr>\n"

    rows_html = _row(major_label(brands_ordered),
                     build_major_quarterly(brands_ordered, quarterly_data, quarters),
                     is_major=True)
    for brand in brands_ordered:
        rows_html += _row(brand_label(brand, stale), quarterly_data.get(brand, {}))
    pnote = _partial_note(ctx["partial_quarters"], brands_ordered, quarters)

    return f"""    <div class="sub-title">1b｜季度 Panel Buy 明細（Unit: M pcs）</div>
    <div class="tbl-scroll"><table>
      <thead><tr>{header}</tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>
    <div class="note">註：QoQ 為對前一季變化率（{fill_year} 1Q QoQ 以 {fill_year-1} 4Q 為基期）；(Cur)＝本次填報，含本期預估月份；Major {len(brands_ordered)} 為各品牌加總，QoQ 以加總值計算。{pnote}</div>"""


# ═══ 共用 SVG 折線圖（2a 月度採購 / 5a 庫存週數）═══
def _svg_line_chart(chart, title, unit, aria, x_label_fn, v_max_fn, y_ticks_fn):
    """chart：extract_chart_intermediate 的 chart1 / chart2（用 series_svg、periods、forecast_from_index）。
    forecast_from_index 之後的點以虛線繪製（與 chart_data 的 (F) 定義一致）。"""
    W, H = 720, 400
    PAD_L, PAD_R, PAD_T, PAD_B = 46, 92, 40, 56
    CHART_W = W - PAD_L - PAD_R
    CHART_H = H - PAD_T - PAD_B
    periods = chart["periods"]
    series = chart["series_svg"]
    ffi = chart["forecast_from_index"]
    if not periods:
        return f"<!-- No data for SVG chart: {title} -->"
    all_vals = [v for s in series for v in s["data"] if v]
    if not all_vals:
        return f"<!-- No values for SVG chart: {title} -->"
    v_max = v_max_fn(max(all_vals))
    v_min = 0

    def x_pos(idx):
        return PAD_L + (idx / max(len(periods) - 1, 1)) * CHART_W

    def y_pos(val):
        if val is None or v_max == v_min:
            return PAD_T + CHART_H
        return PAD_T + CHART_H - ((val - v_min) / (v_max - v_min)) * CHART_H

    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="100%" viewBox="0 0 {W} {H}" role="img" aria-label="{aria}" style="position:absolute;top:0;left:0;display:block;">']
    parts.append(f'<text x="{W/2}" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#1A1A2E">{title}</text>')
    parts.append(f'<text x="{PAD_L}" y="24" text-anchor="start" font-size="11" fill="#898781">Unit: {unit}</text>')

    for yv, txt in y_ticks_fn(v_max):
        yp = y_pos(yv)
        dash = ' stroke-dasharray="3 3"' if yv > 0 else ''
        parts.append(f'<line x1="{PAD_L}" y1="{yp:.1f}" x2="{W-PAD_R}" y2="{yp:.1f}" stroke="#E1E0D9"{dash} stroke-width="1"/>')
        parts.append(f'<text x="{PAD_L-6}" y="{yp+4:.1f}" text-anchor="end" font-size="11" fill="#898781">{txt}</text>')

    # 預估區底色
    if ffi is not None and ffi < len(periods):
        x0 = x_pos(ffi) if ffi == 0 else (x_pos(ffi - 1) + x_pos(ffi)) / 2
        parts.append(f'<rect x="{x0:.1f}" y="{PAD_T}" width="{W-PAD_R-x0:.1f}" height="{CHART_H}" fill="#F3F4F6" opacity="0.7"/>')
        parts.append(f'<text x="{W-PAD_R-4}" y="{PAD_T+12}" text-anchor="end" font-size="10" fill="#898781">(F) 預估</text>')

    for i, (yr, mo) in enumerate(periods):
        if i % 3 == 0 or i == len(periods) - 1:
            label = x_label_fn(yr, mo, i)
            parts.append(f'<text x="{x_pos(i):.1f}" y="{H-PAD_B+18}" text-anchor="middle" font-size="11" fill="#898781">{label}</text>')

    for s in series:
        color = s["color"]
        pts = [(i, x_pos(i), y_pos(v), v) for i, v in enumerate(s["data"]) if v is not None]
        if len(pts) < 2:
            continue
        if ffi is None:
            actual, fcst = pts, []
        else:
            actual = [p for p in pts if p[0] < ffi]
            fcst = [p for p in pts if p[0] >= ffi]
            if actual and fcst:
                fcst = [actual[-1]] + fcst   # 虛線從最後一個實際點接出去
        if len(actual) >= 2:
            parts.append(f'<polyline points="{" ".join(f"{x:.1f},{y:.1f}" for _, x, y, _ in actual)}" fill="none" stroke="{color}" stroke-width="2" stroke-linejoin="round"/>')
        if len(fcst) >= 2:
            parts.append(f'<polyline points="{" ".join(f"{x:.1f},{y:.1f}" for _, x, y, _ in fcst)}" fill="none" stroke="{color}" stroke-width="2" stroke-dasharray="5 4" stroke-linejoin="round"/>')
        _, lx, ly, lv = pts[-1]
        parts.append(f'<circle cx="{lx:.1f}" cy="{ly:.1f}" r="3.5" fill="{color}" stroke="#FCFCFB" stroke-width="2"/>')
        parts.append(f'<text x="{lx+8:.1f}" y="{ly+4:.1f}" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">{s["name"]} {lv:.1f}</text>')

    leg_y = H - 14
    leg_x = 86
    for s in series:
        parts.append(f'<line x1="{leg_x}" y1="{leg_y}" x2="{leg_x+18}" y2="{leg_y}" stroke="{s["color"]}" stroke-width="3"/>')
        parts.append(f'<text x="{leg_x+24}" y="{leg_y+4}" font-size="11" fill="#52514E">{s["name"]}</text>')
        leg_x += 90
    parts.append('</svg>')
    return '\n'.join(parts)


def _build_svg_chart(ctx):
    """Section 2a：品牌月度面板採購趨勢。"""
    c1 = ctx["chart"]["chart1"]
    ffi = c1["forecast_from_index"]
    return _svg_line_chart(
        c1, "品牌月度面板採購趨勢（MI3.0）", "M pcs", "品牌月度面板採購趨勢",
        x_label_fn=lambda yr, mo, i: f"{yr % 100}-{calendar.month_abbr[mo]}" + ("(F)" if (ffi is not None and i >= ffi) else ""),
        v_max_fn=lambda m: m * 1.15,
        y_ticks_fn=lambda v_max: [(v_max * i / 4, f"{v_max * i / 4:.1f}") for i in range(5)],
    )


def _build_inv_svg_chart(ctx):
    """Section 5a：主要品牌庫存週數趨勢。"""
    return _svg_line_chart(
        ctx["chart"]["chart2"], "主要品牌庫存週數趨勢（MI3.0）", "week", "主要品牌庫存週數趨勢",
        x_label_fn=lambda yr, mo, i: f"{yr % 100}-{calendar.month_abbr[mo]}",
        v_max_fn=lambda m: max(m * 1.1, 26),   # at least 26w for context
        y_ticks_fn=lambda v_max: [(yv, str(yv)) for yv in range(0, int(v_max) + 1, 5)],
    )


# ═══ Section 2b: MoM% 變化表 ═══
def _mom_periods(brands_ordered, monthly_series):
    all_periods = set()
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    _sorted = sorted(all_periods)
    # 第一欄 MoM 以視窗前一個月為基期，避免首欄整欄為 "-"
    return _sorted[-9:], (_sorted[-10] if len(_sorted) >= 10 else None)


def _mom_cells_for(series_dict, periods, base_period):
    """回傳 [pct or None]，長度＝periods。"""
    out = []
    prev_val = series_dict.get(base_period) if base_period else None
    for p in periods:
        val = series_dict.get(p)
        out.append((val - prev_val) / prev_val * 100 if (val is not None and prev_val) else None)
        prev_val = val
    return out


def _build_section2b(ctx):
    brands_ordered, monthly_series, stale = ctx["brands_ordered"], ctx["monthly_series"], ctx["stale"]
    fill_year, fill_month = ctx["fill_year"], ctx["fill_month"]
    cur_period = _period(fill_year, fill_month)
    periods, base_period = _mom_periods(brands_ordered, monthly_series)

    header = "<th>MoM</th>"
    for yr, mo in periods:
        header += f"<th>{_month_label(yr, mo)}{'(F)' if yr * 100 + mo >= cur_period else ''}</th>"

    def _row(label, pcts, bold=True):
        b, be = ("<strong>", "</strong>") if bold else ("", "")
        cells = f'<td class="td-txt">{b}{label}{be}</td>'
        for pct in pcts:
            cells += f'<td class="td-num">{_tag_pct(pct)}</td>'
        return f"          <tr>{cells}</tr>\n"

    major = build_major_mom(brands_ordered, monthly_series, periods, base_period)
    rows_html = _row(major_label(brands_ordered), [major[p] for p in periods])
    for brand in brands_ordered:
        bd = {(e["year"], e["month"]): e["value"] for e in monthly_series.get(brand, [])}
        rows_html += _row(brand_label(brand, stale), _mom_cells_for(bd, periods, base_period))

    return f"""    <div class="sub-title">2b｜MoM% 變化表（近 9 個月）</div>
    <div class="tbl-scroll"><table>
      <thead><tr>{header}</tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>
    <div class="note">註：MoM% 基於各月填報之 Panel Buy 預測總量變化；(F)＝本期預估月份。Major 列每月只加總前後兩月都有數字的品牌（同口徑）。</div>"""


# ═══ Section 3: 業務訊息整體摘要 ═══
def _build_section3(ctx):
    app_summary = ctx["app_summary"]
    if not app_summary:
        return ""
    fields = [
        ("整體市況", app_summary.get("summary_futureOutlook") or "（無填寫）"),
        ("產品技術", app_summary.get("summary_techProduct") or "（無填寫）"),
        ("價格相關", app_summary.get("summary_pricing") or "（無填寫）"),
        ("其他焦點", app_summary.get("summary_otherInfo") or "（無填寫）"),
    ]
    rows = ""
    for label, content in fields:
        rows += f'        <tr><td class="facet-label">{label}</td><td class="td-txt">{content}</td></tr>\n'
    return f"""  <div class="section-card">
    <div class="section-title"><div class="section-badge">3</div><h2>業務訊息整體摘要</h2></div>
    <table><tbody>
{rows}    </tbody></table>
  </div>"""


# ═══ Section 4: 各品牌市場訊息矩陣 ═══
def _brand_summary_map(ctx):
    display_map = APP_CONFIG.get(ctx["app_code"], {}).get("brand_display", {})
    return {display_map.get(r["brand_code"], r["brand_code"]): r.asDict() for r in ctx["brand_rows"]}


def _build_section4(ctx):
    brands_ordered, stale = ctx["brands_ordered"], ctx["stale"]
    brand_map = _brand_summary_map(ctx)
    cur_label = _period_label(_period(ctx["fill_year"], ctx["fill_month"]))
    rows_html = ""
    for brand in brands_ordered:
        r = brand_map.get(brand, {})
        na = (f'<span style="color:#C00000">（{cur_label} 未填報）</span>' if (brand in stale and not r)
              else '<span style="color:#999">（無填寫）</span>')
        outlook = r.get("summary_futureOutlook") or na
        tech = r.get("summary_techProduct") or na
        price = r.get("summary_pricing") or na
        other = r.get("summary_otherInfo") or na
        rows_html += f'          <tr><td class="td-txt"><strong>{brand_label(brand, stale)}</strong></td><td class="td-txt">{outlook}</td><td class="td-txt">{tech}</td><td class="td-txt">{price}</td><td class="td-txt">{other}</td></tr>\n'
    return f"""  <div class="section-card">
    <div class="section-title"><div class="section-badge">4</div><h2>各品牌市場訊息矩陣</h2></div>
    <div class="tbl-scroll"><table>
      <thead><tr><th style="width:64px;">品牌</th><th style="width:26%;">市場展望</th><th style="width:26%;">產品技術</th><th style="width:23%;">價格</th><th>其他資訊</th></tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>
  </div>"""


# ═══ Section 5: 庫存週數（5a SVG + 5b 燈號表）═══
def _inv_rows(ctx):
    """5b 列資料：[(brand, weeks, color, lamp_txt, desc)]；缺填品牌沿用其 as_of 的週數，desc 加註。"""
    out = []
    for brand in ctx["brands_ordered"]:
        d = ctx["yearly_data"].get(brand, {})
        weeks = d.get("inv_weeks")
        if weeks is None:
            continue
        color, lamp_txt, desc = get_inv_lamp(ctx["app_code"], brand, weeks)
        if brand in ctx["stale"] and ctx["stale"][brand]:
            desc += f"（沿用 {_period_label(ctx['stale'][brand])}）"
        out.append((brand, weeks, color, lamp_txt, desc))
    return out


def _build_section5(ctx):
    fill_year, fill_month, app_code = ctx["fill_year"], ctx["fill_month"], ctx["app_code"]
    month_abbr = calendar.month_abbr[fill_month]
    svg_chart = _build_inv_svg_chart(ctx)
    rows_html = ""
    for brand, weeks, color, lamp_txt, desc in _inv_rows(ctx):
        rows_html += f'          <tr><td style="text-align:center;"><span class="lamp" style="background:{color};"></span><strong style="color:{color};">{lamp_txt}</strong></td><td class="td-txt"><strong>{brand_label(brand, ctx["stale"])}</strong></td><td class="td-num"><strong>{_fmt_weeks(weeks)}</strong></td><td class="td-txt">{desc}</td></tr>\n'
    return f"""  <div class="section-card">
    <div class="section-title"><div class="section-badge">5</div><h2>主要品牌庫存週數（{month_abbr}'{fill_year % 100} pipeline）</h2></div>
    <div class="sub-title">5a｜MI3.0 庫存週數趨勢（Unit: week）</div>
    <div class="chart-wrap">
{svg_chart}
    </div>
    <div class="sub-title">5b｜{month_abbr}'{fill_year % 100} 庫存燈號</div>
    <table>
      <thead><tr><th style="width:64px;">燈號</th><th style="width:110px;">品牌</th><th style="width:90px;">週數</th><th class="td-txt">判讀</th></tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table>
    <div class="note">{get_inv_note(app_code, ctx["brands_ordered"])}</div>
  </div>"""


# ═══════════ AI Insight（三段，html 與 mail 共用同一份文字）═══════════
def generate_insights(ctx):
    """回傳 {"1a": str, "1b": str, "2": str}。prompt 明示報告月份、欄位定義、(F)/沿用/'-' 的意義與月份標籤。"""
    brands_ordered, yearly_data, stale = ctx["brands_ordered"], ctx["yearly_data"], ctx["stale"]
    fill_year, fill_month = ctx["fill_year"], ctx["fill_month"]
    cur_label = _period_label(_period(fill_year, fill_month))
    stale_hint = ("；本月未填、數字沿用上期的品牌：" + "、".join(
        f"{b}（{_period_label(a) if a else '無'}）" for b, a in stale.items())) if stale else ""

    # 1a 年度
    head = (f"報告月份：{fill_year} 年 {fill_month} 月（{cur_label}）。欄位：Y{(fill_year-1) % 100}＝{fill_year-1} 全年；"
            f"Y{fill_year % 100}(Pre)＝上期填報的 {fill_year} 全年預估；Y{fill_year % 100}(Cur)＝本期填報的 {fill_year} 全年預估（{fill_year} 是本年度，不是明年）；"
            f"YoY＝Y{fill_year % 100}(Cur) 對 Y{(fill_year-1) % 100}；YTD＝1~{fill_month-1} 月實績加總；Hit%＝YTD / Y{fill_year % 100}(Cur)。單位 M pcs{stale_hint}。\n")
    lines = []
    for b in brands_ordered:
        d = yearly_data.get(b, {})
        if d.get("y_cur") is None:
            lines.append(f"{b}: 本期無年度數字")
            continue
        lines.append(
            f"{b}{'（沿用）' if b in stale else ''}: Y{(fill_year-1) % 100}={_fmt_num(d.get('y_prev'))}, "
            f"Pre={_fmt_num(d.get('y_cur_pre'))}, Cur={_fmt_num(d.get('y_cur'))}, YoY={_fmt_pct(d.get('y_cur_yoy'))}, "
            f"YTD={_fmt_num(d.get('ytd'))}, Hit={_fmt_num(d.get('hit_pct'), 0)}%")
    insight_1a = generate_insight("年度 Panel Buy 分析", head + "; ".join(lines))

    # 1b 季度
    cur_q, quarters = _quarters_for(fill_year, fill_month)
    qhead = (f"【重要約束】表格資料僅到 {fill_year} {cur_q}Q 為止，請只分析有數字的季度，絕對不要提及或推測之後的季度。"
             f"季度值＝三個月加總；{fill_year % 100} {cur_q}Q (Cur) 含本期預估月份；'-' 代表該季月份未填齊，不要解讀為衰退{stale_hint}。\n")
    qlines = []
    for b in brands_ordered[:4]:
        bq = ctx["quarterly_data"].get(b, {})
        qlines.append(f"{b}: " + ", ".join(f"{_q_label(yr, q)}={_fmt_num(bq.get((yr, q)))}" for yr, q in quarters))
    insight_1b = generate_insight("季度 Panel Buy 分析", qhead + "; ".join(qlines))

    # 2 月度（近三個月，帶月份標籤與 (F)）
    c1 = ctx["chart"]["chart1"]
    idx = list(range(max(0, len(c1["periods"]) - 3), len(c1["periods"])))
    labels = [f"{calendar.month_abbr[c1['periods'][i][1]]}'{c1['periods'][i][0] % 100}"
              + ("(F)" if (c1["forecast_from_index"] is not None and i >= c1["forecast_from_index"]) else "") for i in idx]
    mhead = (f"報告月份 {cur_label}；以下為各品牌近三個月（{labels[0]}~{labels[-1]}）的月度 Panel Buy，"
             f"(F)＝本期填報的預估值，'-'＝該月未填（請勿視為 0 或衰退）；單位 M pcs{stale_hint}。\n")
    mlines = []
    for s in c1["series"]:
        mlines.append(f"{s['name']}{'（沿用）' if s['name'] in stale else ''}: " +
                      ", ".join(f"{labels[k]}={_fmt_num(s['data'][i], 2)}" for k, i in enumerate(idx)))
    insight_2 = generate_insight("月度採購趨勢分析", mhead + "; ".join(mlines))
    return {"1a": insight_1a, "1b": insight_1b, "2": insight_2}


# ═══════════ 組裝完整 HTML ═══════════
def generate_html(ctx):
    """Generate complete HTML report (web, with SVG)."""
    app_code, fill_year, fill_month = ctx["app_code"], ctx["fill_year"], ctx["fill_month"]
    month_abbr = calendar.month_abbr[fill_month]
    title = report_title(app_code, fill_year, fill_month)
    link_url, link_txt = dashboard_link(app_code)
    link_html = (f'    <div class="header-link"><a href="{link_url}" target="_blank">{link_txt}</a></div>'
                 if link_url else '')
    banner = missing_banner_text(ctx["stale"], ctx["partial_quarters"], fill_year, fill_month)
    banner_html = f'\n  <div class="warn-box">&#9888;&#65039; {banner}</div>' if banner else ""
    ins = ctx["insights"]

    s1a = _build_section1a(ctx)
    s1b = _build_section1b(ctx)
    svg_chart = _build_svg_chart(ctx)
    s2b = _build_section2b(ctx)
    s3 = _build_section3(ctx)
    s4 = _build_section4(ctx)
    s5 = _build_section5(ctx)

    return f"""<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>{title}</title>
{HTML_CSS}
</head>
<body>
<div class="container">

  <!-- Header -->
  <div class="header">
    <h1>{title}</h1>
    <div class="header-meta">
      <span>&#128197; 業務資訊蒐集｜Panel Buy / 市場訊息 / 庫存觀測</span>
      <span>&#128202; 資料來源：MI DataHub / MI3.0</span>
      <span>&#128337; 數據截至：{month_abbr}'{fill_year % 100} 填報</span>
    </div>
{link_html}
  </div>

  <!-- 頂部提示 -->
  <div style="background:#FFF8E1;border:1px solid #FFD54F;border-radius:8px;padding:12px 18px;margin-bottom:18px;font-size:13px;color:#5D4037;line-height:1.7;">
    &#128196; 完整圖表版報告已隨信附上（.html 附件），請以瀏覽器開啟以檢視互動圖表。
  </div>{banner_html}

  <!-- Section 1 -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">1</div><h2>品牌年度／季度 Panel buy 分析</h2></div>
{s1a}
    <div class="insight-box">&#128204; <strong>AI 年度分析：</strong>{ins["1a"]}</div>
{s1b}
    <div class="insight-box">&#128204; <strong>AI 季度分析：</strong>{ins["1b"]}</div>
  </div>

  <!-- Section 2 -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">2</div><h2>品牌月度面板採購</h2></div>
    <div class="sub-title">2a｜MI3.0 月度採購趨勢（Unit: M pcs）</div>
    <div class="chart-wrap">
{svg_chart}
    </div>
    <div class="insight-box">&#128204; <strong>趨勢判讀：</strong>{ins["2"]}</div>
{s2b}
  </div>

{s3}

{s4}

{s5}

  <!-- Footer -->
  <div class="footer">
    本報告由 MI DataHub / myAgent 自動彙整產出，數據截至 {month_abbr}'{fill_year % 100} 填報；互動篩選請至 MI3.0 系統。
  </div>

</div>
</body>
</html>"""


# ════════════════════════════════════════════════════════════
# Step 3b: generate_mail_body — Email 內文 HTML
# ════════════════════════════════════════════════════════════

def _mail_tag(v):
    """Return email-safe percentage tag."""
    if v is None:
        return "-"
    txt = f"{v:+.0f}%" if v != 0 else "0%"
    if v > 0:
        return f'<span class=p>{txt}</span>'
    elif v < 0:
        return f'<span class=n>{txt}</span>'
    return f'<span class=u>{txt}</span>'


def _mail_section_start(num, title):
    return f'''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>{num}</td><td class=h>{title}</td></tr></table>'''


def _mail_section_end():
    return '</td></tr></table></td></tr>\n<tr><td class=s height=14></td></tr>'


def generate_mail_body(ctx):
    """Generate Email-compatible HTML (table-based, 640px).
    Keeps {{CHART1_PNG_URL}} and {{CHART2_PNG_URL}} placeholders for downstream PNG replacement.
    Returns minified HTML string."""
    import re as _re
    app_code, fill_year, fill_month = ctx["app_code"], ctx["fill_year"], ctx["fill_month"]
    brands_ordered, yearly_data, stale = ctx["brands_ordered"], ctx["yearly_data"], ctx["stale"]
    quarterly_data, monthly_series = ctx["quarterly_data"], ctx["monthly_series"]
    ins = ctx["insights"]
    month_abbr = calendar.month_abbr[fill_month]
    cur_period = _period(fill_year, fill_month)
    title = report_title(app_code, fill_year, fill_month)
    link_url, link_txt = dashboard_link(app_code)
    link_html = (f'<br><a href="{link_url}" style="font-size:13px;font-weight:700;color:#FFFFFF;text-decoration:underline">{link_txt}</a>'
                 if link_url else '')

    css = '''<style>
body{margin:0;background:#F3F4F6}
table{border-collapse:collapse}
td,th{font-family:'Microsoft JhengHei','Noto Sans TC',sans-serif;font-size:12px}
th{background:#1F4E79;color:#FFFFFF;padding:5px 3px;font-size:11px}
td{padding:5px 4px;border-bottom:1px solid #DCE6F0;text-align:right}
.t{text-align:left}
.ce{text-align:center}
.w{padding:18px 22px;border:0;text-align:left}
.s{padding:0;border:0;font-size:0;line-height:0}
.b{background:#1F4E79;color:#FFFFFF;font-weight:700;text-align:center;border-radius:50%;padding:0;border:0}
.h{font-size:16px;font-weight:700;padding:0 0 0 10px;border:0;text-align:left}
.st{color:#1F4E79;font-weight:700;font-size:13px;padding:14px 0 7px;border:0;text-align:left}
.i{background:#F5F7FA;padding:10px 14px;line-height:1.8;border:0;text-align:left}
.no{font-size:11px;color:#8A8A96;padding:6px 0 0;border:0;text-align:left;line-height:1.6}
.fl{background:#EDF2F7;color:#1F4E79;font-weight:700;text-align:left;vertical-align:top;width:76px}
.p{background:#E8F5E9;color:#2E7D32;border-radius:8px;padding:0 5px;font-size:11px}
.n{background:#FDECEC;color:#C00000;border-radius:8px;padding:0 5px;font-size:11px}
.u{background:#FFF8E1;color:#946C00;border-radius:8px;padding:0 5px;font-size:11px}
.f{font-size:11px;color:#9E9E9E;text-align:center;line-height:1.7;padding:12px 0 20px;border:0}
</style>'''

    parts = []
    parts.append(f'''<!DOCTYPE html>
<html xmlns:o="urn:schemas-microsoft-com:office:office" lang=zh-Hant><head><meta charset=UTF-8><title>{title}</title>
<!--[if mso]><noscript><xml><o:OfficeDocumentSettings><o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml></noscript><![endif]-->
{css}</head>
<body bgcolor=#F3F4F6>
<table width=100% cellpadding=0 cellspacing=0 bgcolor=#F3F4F6><tr><td class=s align=left>
<table width=640 cellpadding=0 cellspacing=0 align=left>
<tr><td class=s height=16></td></tr>''')

    # Header
    parts.append(f'''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#1F4E79><tr><td class=w style="padding:22px 24px">
<span style="font-size:22px;font-weight:700;color:#FFFFFF">{title}</span><br>
<span style="font-size:12px;color:#CFE0F0;line-height:2">&#128197; 業務資訊蒐集｜Panel Buy / 市場訊息 / 庫存觀測　&#128202; 資料來源：MI DataHub / MI3.0　&#128337; 數據截至：{month_abbr}\'{fill_year % 100} 填報</span>
{link_html}
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>''')

    # Top notice
    parts.append('''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFF8E1 style="border:1px solid #FFD54F"><tr><td class=w style="padding:10px 16px;color:#5D4037;line-height:1.7">&#128196; 完整圖表版報告已隨信附上（.html 附件），請以瀏覽器開啟以檢視互動圖表。</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>''')

    # 缺填警示
    banner = missing_banner_text(stale, ctx["partial_quarters"], fill_year, fill_month)
    if banner:
        parts.append(f'''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FDECEC style="border:1px solid #F5B5B5"><tr><td class=w style="padding:10px 16px;color:#8B1A1A;line-height:1.7">&#9888;&#65039; {banner}</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>''')

    # ── Section 1: 年度／季度 Panel Buy ──
    parts.append(_mail_section_start(1, '品牌年度／季度 Panel buy 分析'))
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>1a｜年度 Panel Buy 總表（Unit: M pcs）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    _major_1a = build_major_yearly(brands_ordered, yearly_data, app_code)
    show_prev_yoy = any(d.get('y_prev_yoy') is not None
                        for d in [_major_1a] + [yearly_data.get(b, {}) for b in brands_ordered])
    parts.append(f'<tr><th>Brand</th><th>Y25</th><th>Y26 (Pre)</th><th>Y26 (Cur)</th>{"<th>Y25 YoY%</th>" if show_prev_yoy else ""}<th>Y26 (Pre) YoY%</th><th>Y26 YoY%</th><th>YTD (1~{fill_month-1}M)</th><th>Hit%</th></tr>')
    _rows_1a = [(major_label(brands_ordered), _major_1a, True)]
    _rows_1a += [(brand_label(b, stale), yearly_data.get(b, {}), False) for b in brands_ordered]
    for ri, (_label, d, _is_major) in enumerate(_rows_1a):
        bg = ' bgcolor=#EDF2F7' if _is_major else (' bgcolor=#EFF5FB' if ri % 2 == 1 else '')
        hit = f"{d.get('hit_pct'):.0f}%" if d.get('hit_pct') is not None else '-'
        prev_yoy_td = f'<td>{_mail_tag(d.get("y_prev_yoy"))}</td>' if show_prev_yoy else ''
        parts.append(f'<tr{bg}><td class=t><b>{_label}</b></td><td>{_fmt_num(d.get("y_prev"))}</td><td>{_fmt_num(d.get("y_cur_pre"))}</td><td><b>{_fmt_num(d.get("y_cur"))}</b></td>{prev_yoy_td}<td>{_mail_tag(d.get("y_cur_pre_yoy"))}</td><td>{_mail_tag(d.get("y_cur_yoy"))}</td><td>{_fmt_num(d.get("ytd"))}</td><td>{hit}</td></tr>')
    parts.append('</table>')
    snote = stale_note_text(stale, fill_year, fill_month)
    if snote:
        parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>{snote}</td></tr></table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>AI 年度分析：</b>{ins["1a"] or ""}</td></tr></table>')

    # 1b quarterly table
    cur_q, quarters = _quarters_for(fill_year, fill_month)
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>1b｜季度 Panel Buy 明細（Unit: M pcs）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    hdr = '<th>Brand</th>' + ''.join(f'<th>{_q_label(yr, q)}{" (Cur)" if yr==fill_year and q==cur_q else ""}</th>' for yr, q in quarters)
    hdr += ''.join(f'<th>{fill_year%100} {q}Q QoQ</th>' for q in range(1, cur_q + 1))
    parts.append(f'<tr>{hdr}</tr>')
    _rows_1b = [(major_label(brands_ordered),
                 build_major_quarterly(brands_ordered, quarterly_data, quarters), True)]
    _rows_1b += [(brand_label(b, stale), quarterly_data.get(b, {}), False) for b in brands_ordered]
    for ri, (_label, bq, _is_major) in enumerate(_rows_1b):
        bg = ' bgcolor=#EDF2F7' if _is_major else (' bgcolor=#EFF5FB' if ri % 2 == 1 else '')
        cells = f'<td class=t><b>{_label}</b></td>'
        qvals = []
        for yr, q in quarters:
            v = bq.get((yr, q))
            qvals.append(v)
            cells += f'<td><b>{_fmt_num(v)}</b></td>' if (yr == fill_year and q == cur_q) else f'<td>{_fmt_num(v)}</td>'
        for qi in range(len(quarters) - cur_q, len(quarters)):
            cv = qvals[qi]
            pv = qvals[qi - 1] if qi > 0 else None
            qoq = ((cv - pv) / pv * 100) if (cv and pv and pv != 0) else None
            cells += f'<td>{_mail_tag(qoq)}</td>'
        parts.append(f'<tr{bg}>{cells}</tr>')
    parts.append('</table>')
    pnote = _partial_note(ctx["partial_quarters"], brands_ordered, quarters)
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>註：QoQ 為對前一季變化率（{fill_year} 1Q QoQ 以 {fill_year-1} 4Q 為基期）；(Cur)＝本次填報，含本期預估月份；Major {len(brands_ordered)} 為各品牌加總，QoQ 以加總值計算。{pnote}</td></tr></table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>AI 季度分析：</b>{ins["1b"] or ""}</td></tr></table>')
    parts.append(_mail_section_end())

    # ── Section 2: 月度採購 ──
    parts.append(_mail_section_start(2, '品牌月度面板採購'))
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>2a｜MI3.0 月度採購趨勢（Unit: M pcs）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0 bgcolor=#FCFCFB style="border:1px solid #E0E0E0"><tr><td class=s align=center><img src="{{CHART1_PNG_URL}}" width=590 height=328 alt="四品牌月度面板採購趨勢（MI3.0），Unit: M pcs" style="display:block;border:0;max-width:100%"></td></tr></table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>趨勢判讀：</b>{ins["2"] or ""}</td></tr></table>')

    # 2b MoM table
    periods, base_period = _mom_periods(brands_ordered, monthly_series)
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>2b｜MoM% 變化表（近 9 個月）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    mom_hdr = '<th>MoM</th>' + ''.join(f'<th>{_month_label(yr, mo)}{"(F)" if yr * 100 + mo >= cur_period else ""}</th>' for yr, mo in periods)
    parts.append(f'<tr>{mom_hdr}</tr>')
    major = build_major_mom(brands_ordered, monthly_series, periods, base_period)
    parts.append('<tr><td class=t><b>' + major_label(brands_ordered) + '</b></td>' + ''.join(f'<td>{_mail_tag(major[p])}</td>' for p in periods) + '</tr>')
    for ri, brand in enumerate(brands_ordered):
        bg = ' bgcolor=#EFF5FB' if ri % 2 == 0 else ''
        bd = {(e["year"], e["month"]): e["value"] for e in monthly_series.get(brand, [])}
        cells = ''.join(f'<td>{_mail_tag(p)}</td>' for p in _mom_cells_for(bd, periods, base_period))
        parts.append(f'<tr{bg}><td class=t><b>{brand_label(brand, stale)}</b></td>{cells}</tr>')
    parts.append('</table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>註：MoM% 基於各月填報之 Panel Buy 預測總量變化；(F)＝本期預估月份。Major 列每月只加總前後兩月都有數字的品牌（同口徑）。</td></tr></table>')
    parts.append(_mail_section_end())

    # ── Section 3: 業務訊息整體摘要 ──
    app_summary = ctx["app_summary"]
    if app_summary:
        parts.append(_mail_section_start(3, '業務訊息整體摘要'))
        parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=10 colspan=2></td></tr>')
        fields = [
            ("整體市況", app_summary.get("summary_futureOutlook") or "（無填寫）"),
            ("產品技術", app_summary.get("summary_techProduct") or "（無填寫）"),
            ("價格相關", app_summary.get("summary_pricing") or "（無填寫）"),
            ("其他焦點", app_summary.get("summary_otherInfo") or "（無填寫）"),
        ]
        for fi, (label, content) in enumerate(fields):
            bg = ' bgcolor=#EFF5FB' if fi % 2 == 1 else ''
            parts.append(f'<tr{bg}><td class=fl>{label}</td><td class=t style="vertical-align:top">{content}</td></tr>')
        parts.append('</table>')
        parts.append(_mail_section_end())

    # ── Section 4: 各品牌市場訊息矩陣 ──
    brand_map = _brand_summary_map(ctx)
    cur_label = _period_label(cur_period)
    parts.append(_mail_section_start(4, '各品牌市場訊息矩陣'))
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=10 colspan=5></td></tr>')
    parts.append('<tr><th width=44>品牌</th><th width=145>市場展望</th><th width=145>產品技術</th><th width=130>價格</th><th>其他資訊</th></tr>')
    for ri, brand in enumerate(brands_ordered):
        bg = ' bgcolor=#EFF5FB' if ri % 2 == 1 else ''
        r = brand_map.get(brand, {})
        na = (f'<span style="color:#C00000">（{cur_label} 未填報）</span>' if (brand in stale and not r)
              else '<span style="color:#9E9E9E">（無填寫）</span>')
        outlook = r.get("summary_futureOutlook") or na
        tech = r.get("summary_techProduct") or na
        price = r.get("summary_pricing") or na
        other = r.get("summary_otherInfo") or na
        parts.append(f'<tr valign=top{bg}><td class=t><b>{brand_label(brand, stale)}</b></td><td class=t>{outlook}</td><td class=t>{tech}</td><td class=t>{price}</td><td class=t>{other}</td></tr>')
    parts.append('</table>')
    parts.append(_mail_section_end())

    # ── Section 5: 庫存週數 ──
    parts.append(_mail_section_start(5, f'主要品牌庫存週數（{month_abbr}\'{fill_year % 100} pipeline）'))
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>5a｜MI3.0 庫存週數趨勢（Unit: week）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0 bgcolor=#FCFCFB style="border:1px solid #E0E0E0"><tr><td class=s align=center><img src="{{CHART2_PNG_URL}}" width=590 height=328 alt="主要品牌庫存週數趨勢（MI3.0），Unit: week" style="display:block;border:0;max-width:100%"></td></tr></table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>5b｜{month_abbr}\'{fill_year % 100} 庫存燈號</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    parts.append('<tr><th width=60>燈號</th><th width=90>品牌</th><th width=64>週數</th><th>判讀</th></tr>')
    for ri, (brand, weeks, color, lamp_txt, desc) in enumerate(_inv_rows(ctx)):
        bg = ' bgcolor=#EFF5FB' if ri % 2 == 1 else ''
        parts.append(f'<tr{bg}><td class=ce style="color:{color}">&#9679; <b>{lamp_txt}</b></td><td class=t><b>{brand_label(brand, stale)}</b></td><td><b>{_fmt_weeks(weeks)}</b></td><td class=t>{desc}</td></tr>')
    parts.append('</table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>{get_inv_note(app_code, brands_ordered)}</td></tr></table>')
    parts.append(_mail_section_end())

    # Footer
    parts.append(f'<tr><td class=f>本報告由 MI DataHub / myAgent 自動彙整產出，數據截至 {month_abbr}\'{fill_year % 100} 填報。<br>本信為 Outlook 表格降級版；完整圖表版請開啟隨信 .html 附件。互動篩選請至 MI3.0 系統。</td></tr>')
    parts.append('</table>\n</td></tr></table>\n</body></html>')

    raw_html = '\n'.join(parts)
    return _re.sub(r'\s+', ' ', raw_html).strip()


# ════════════════════════════════════════════════════════════
# Step 3c: generate_chart_data — 圖表數據 JSON（schema 不變）
# ════════════════════════════════════════════════════════════

def generate_chart_data(app_code, brief_month, chart_intermediate):
    """Generate chart_data JSON string per spec 3c schema.
    chart_intermediate comes from extract_chart_intermediate(); 只輸出 series（前 4 名），不輸出 series_svg / periods。"""
    c1 = chart_intermediate["chart1"]
    c2 = chart_intermediate["chart2"]
    payload = {
        "app_name": app_code,
        "brief_month": brief_month,
        "charts": [
            {
                "id": "CHART1",
                "placeholder": "{{CHART1_PNG_URL}}",
                "chart_type": "line",
                "title": "四品牌月度面板採購趨勢（MI3.0）",
                "unit": "M pcs",
                "x_labels": c1["x_labels"],
                "forecast_from_index": c1["forecast_from_index"],
                "series": [{"name": s["name"], "color": s["color"], "data": s["data"]} for s in c1["series"]],
            },
            {
                "id": "CHART2",
                "placeholder": "{{CHART2_PNG_URL}}",
                "chart_type": "line",
                "title": "主要品牌庫存週數趨勢（MI3.0）",
                "unit": "week",
                "x_labels": c2["x_labels"],
                "forecast_from_index": c2["forecast_from_index"],
                "series": [{"name": s["name"], "color": s["color"], "data": s["data"]} for s in c2["series"]],
            },
        ],
    }
    return json.dumps(payload, ensure_ascii=False, separators=(',', ':'))


print("✅ HTML 產出函式已定義（完整 web 版，含 SVG + AI Insight + mail_body + chart_data）")


In [ ]:
# [c06] run_merge
from pyspark.sql import Row
from datetime import datetime
import re as _re


def build_report_context(app_code, fill_year, fill_month):
    """取數一次、組成 ctx 給 generate_html / generate_mail_body / generate_chart_data 共用
    （v1 在 generate_html 與 c06 各取一次，兩邊可能不同步）。"""
    print("  → 取品牌名單 / 年度 / 季度 / 月度 / 庫存 / 摘要...")
    brands_ordered = get_brand_order(app_code, fill_year, fill_month, BRAND_LOOKBACK_MONTHS)
    yearly_data = get_yearly_data(app_code, fill_year, fill_month, BRAND_LOOKBACK_MONTHS)
    quarterly_data, partial_quarters = get_quarterly_data(app_code, fill_year, fill_month)
    monthly_series = get_monthly_series(app_code, fill_year, fill_month)
    inv_series = get_inventory_series(app_code, fill_year, fill_month)
    app_summary, brand_rows = get_summary_data(app_code, fill_year, fill_month)
    stale = stale_info(brands_ordered, yearly_data, fill_year, fill_month)
    chart = extract_chart_intermediate(brands_ordered, monthly_series, inv_series, fill_year, fill_month, app_code)
    ctx = {
        "app_code": app_code, "fill_year": fill_year, "fill_month": fill_month,
        "brief_month": f"{fill_year}-{fill_month:02d}",
        "brands_ordered": brands_ordered, "yearly_data": yearly_data,
        "quarterly_data": quarterly_data, "partial_quarters": partial_quarters,
        "monthly_series": monthly_series, "inv_series": inv_series,
        "app_summary": app_summary, "brand_rows": brand_rows,
        "stale": stale, "chart": chart,
    }
    print(f"  → 品牌 {brands_ordered}；本月未填 {list(stale) or '無'}")
    print("  → 產生 AI Insight（1a / 1b / 2）...")
    ctx["insights"] = generate_insights(ctx)
    return ctx


def _prev_major_n(app_code, brief_month):
    """從目標表上一期 mail_body 讀出 'Major N' 的 N，供品牌數比對；讀不到回傳 None。"""
    if not spark.catalog.tableExists(TARGET_TABLE):
        return None
    row = spark.sql(f"""
        SELECT mail_body FROM {TARGET_TABLE}
        WHERE app_name = '{app_code}' AND brief_month < '{brief_month}'
        ORDER BY brief_month DESC LIMIT 1
    """).first()
    if not row or not row["mail_body"]:
        return None
    m = _re.search(r"Major (\d+)", row["mail_body"])
    return int(m.group(1)) if m else None


# ── 產出 HTML + mail_body + chart_data 並收集結果 ──
results = []
all_warnings = {}

for app_info in apps_to_process:
    app_code = app_info["app_code"]
    fill_year = app_info["fill_year"]
    fill_month = app_info["fill_month"]
    brief_month = f"{fill_year}-{fill_month:02d}"

    print(f"\n{'='*60}")
    print(f"產出 {app_code} / {brief_month}")
    print(f"{'='*60}")

    try:
        ctx = build_report_context(app_code, fill_year, fill_month)

        html_content = generate_html(ctx)
        print(f"  → html_content: {len(html_content)} chars")
        mail_body = generate_mail_body(ctx)
        print(f"  → mail_body: {len(mail_body)} chars")
        chart_data = generate_chart_data(app_code, brief_month, ctx["chart"])
        print(f"  → chart_data: {len(chart_data)} chars")

        # 內容檢查（只警告，不擋寫入；擋產出由 c02 on_missing_brand 決定）
        warns = content_checks(ctx, prev_major_n=_prev_major_n(app_code, brief_month))
        all_warnings[app_code] = warns
        for w in warns:
            print(f"  ⚠️ {w}")

        results.append({
            "app_name": app_code,
            "brief_month": brief_month,
            "html_content": html_content,
            "created_at": datetime.now(),
            "mail_body": mail_body,
            "chart_data": chart_data,
        })
        print(f"✅ {app_code}/{brief_month} 三欄產出成功（警告 {len(warns)} 項）")
    except Exception as e:
        print(f"❌ {app_code}/{brief_month} 產出失敗: {e}")
        import traceback
        traceback.print_exc()

print(f"\n── 總計產出 {len(results)} 份報告 ──")

# ── 寫入 Delta Table ──
if results:
    schema = StructType([
        StructField("app_name", StringType(), False),
        StructField("brief_month", StringType(), False),
        StructField("html_content", StringType(), False),
        StructField("created_at", TimestampType(), False),
        StructField("mail_body", StringType(), True),
        StructField("chart_data", StringType(), True),
    ])

    df_results = spark.createDataFrame([Row(**r) for r in results], schema=schema)
    df_results.createOrReplaceTempView("_sales_brief_new")

    if not spark.catalog.tableExists(TARGET_TABLE):
        df_results.write.saveAsTable(TARGET_TABLE)
        spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Sales Brief 郵件報告，by 應用/月份（html_content + mail_body + chart_data）'")
        print(f"✅ 首次建立 {TARGET_TABLE}，共 {len(results)} 筆")
    else:
        # MERGE：以 (app_name, brief_month) 為 key，有就更新、沒有就新增（歷史月份保留）
        spark.sql(f"""
            MERGE INTO {TARGET_TABLE} AS t
            USING _sales_brief_new AS s
            ON t.app_name = s.app_name AND t.brief_month = s.brief_month
            WHEN MATCHED THEN UPDATE SET
                t.html_content = s.html_content,
                t.created_at = s.created_at,
                t.mail_body = s.mail_body,
                t.chart_data = s.chart_data
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"✅ 已 MERGE 寫入 {TARGET_TABLE}，本次處理 {len(results)} 筆（歷史月份保留）")
else:
    print("⚠️ 無需產出的報告")

if any(all_warnings.values()):
    print("\n── 內容警告彙整（請人工確認後再寄信）──")
    for app, ws in all_warnings.items():
        for w in ws:
            print(f"  {app}: {w}")


In [ ]:
# [c07] validate
# ════════════════════════════════════════════════════════════
# Step 5: 驗證結果（格式關卡 + 內容關卡）
# 格式關卡不過 → ❌（overall_pass=False）；內容關卡 → ⚠️ 警告，需人工確認
# ════════════════════════════════════════════════════════════
import re

TABLE = "micenter.mi3_datahub_prod.g_sales_brief_mail_report"
ALL_APPS = ["TV", "NB", "MNT"]

df_check = spark.sql(f"""
    SELECT app_name, brief_month,
           LENGTH(html_content) AS html_len,
           LENGTH(mail_body) AS mail_len,
           LENGTH(chart_data) AS chart_len,
           created_at
    FROM {TABLE}
    WHERE app_name IN ('TV', 'NB', 'MNT')
    ORDER BY app_name, brief_month DESC
""")
display(df_check)


def _text_of(html):
    """去 tag 後的純文字（給內容關卡用）。"""
    t = re.sub(r"<style.*?</style>", " ", html or "", flags=re.S)
    t = re.sub(r"<[^>]+>", " ", t)
    return re.sub(r"\s+", " ", t)


overall_pass = True
for APP in ALL_APPS:
    rows = spark.sql(f"""
        SELECT brief_month, html_content, mail_body, chart_data
        FROM {TABLE}
        WHERE app_name = '{APP}'
        ORDER BY brief_month DESC LIMIT 2
    """).collect()

    if not rows:
        print(f"❌ {APP}: 目標表無資料")
        overall_pass = False
        continue

    row = rows[0]
    prev = rows[1] if len(rows) > 1 else None
    html_content = row["html_content"]
    mail_body = row["mail_body"]
    chart_data_str = row["chart_data"]

    print(f"\n{'='*60}")
    print(f"品質驗證關卡 — {APP} / {row['brief_month']}")
    print(f"{'='*60}")
    all_pass = True
    warns = []

    # 1️⃣ 佔位符完整性
    print("\n── 1. 佔位符完整性 ──")
    c1_count = mail_body.count("{{CHART1_PNG_URL}}") if mail_body else 0
    c2_count = mail_body.count("{{CHART2_PNG_URL}}") if mail_body else 0
    other_placeholders = re.findall(r'\{\{[^}]+\}\}', mail_body or '')
    other_placeholders = [p for p in other_placeholders if p not in ('{{CHART1_PNG_URL}}', '{{CHART2_PNG_URL}}')]
    html_placeholders = re.findall(r'\{\{[^}]+\}\}', html_content or '')
    if c1_count == 1 and c2_count == 1 and not other_placeholders and not html_placeholders:
        print("  ✅ mail_body 含 CHART1/CHART2 各一次，無殘留佔位符；html_content 無佔位符")
    else:
        all_pass = False
        print(f"  ❌ CHART1={c1_count}, CHART2={c2_count}, 其他殘留={other_placeholders}, html佔位符={html_placeholders}")

    # 2️⃣ 體積預算
    print("\n── 2. 體積預算 ──")
    mail_len = len(mail_body) if mail_body else 0
    if mail_len <= 35000:
        print(f"  ✅ mail_body = {mail_len:,} chars (≤ 35,000)")
    else:
        all_pass = False
        print(f"  ❌ mail_body = {mail_len:,} chars (超過 35,000 上限!)")

    # 3️⃣ chart JSON 可解析 + 內容
    print("\n── 3. chart JSON ──")
    charts = []
    try:
        cd = json.loads(chart_data_str)
        charts = cd.get("charts", [])
        chart_ok = True
        for chart in charts:
            x_len = len(chart.get("x_labels", []))
            for s in chart.get("series", []):
                if len(s.get("data", [])) != x_len:
                    chart_ok = False
                    print(f"  ❌ {chart['id']}: series '{s['name']}' data len={len(s['data'])} != x_labels len={x_len}")
                if s.get("data") and s["data"][-1] is None:
                    warns.append(f"{chart['id']} {s['name']} 最後一點為 null（該品牌缺最新月份）")
        if len(charts) != c1_count + c2_count:
            chart_ok = False
            print(f"  ❌ charts 數={len(charts)}, mail_body 佔位符數={c1_count + c2_count}")
        # CHART1：當月必須標 (F)（forecast_from_index 指向含 brief_month 的那格）
        c1 = next((c for c in charts if c["id"] == "CHART1"), None)
        if c1:
            ffi = c1.get("forecast_from_index")
            labels = c1.get("x_labels", [])
            mon = calendar.month_abbr[int(row["brief_month"][5:7])]
            cur_idx = next((i for i, l in enumerate(labels) if l.replace("(F)", "").endswith(mon)), None)
            if ffi is None or cur_idx is None or ffi != cur_idx or not labels[cur_idx].endswith("(F)"):
                warns.append(f"CHART1 當月 {mon} 未標 (F)：forecast_from_index={ffi}, labels={labels[-4:]}")
        # 顏色：同品牌在 CHART1 / CHART2 需同色；與上期同色
        color_now = {s["name"]: s["color"] for c in charts for s in c.get("series", [])}
        for c in charts:
            for s in c.get("series", []):
                if color_now[s["name"]] != s["color"]:
                    warns.append(f"{s['name']} 在兩張圖顏色不同")
        if prev and prev["chart_data"]:
            try:
                prev_colors = {s["name"]: s["color"] for c in json.loads(prev["chart_data"]).get("charts", []) for s in c.get("series", [])}
                for b, col in color_now.items():
                    if b in prev_colors and prev_colors[b] != col:
                        warns.append(f"{b} 顏色與上期 {prev['brief_month']} 不同（{prev_colors[b]} → {col}）")
            except Exception:
                pass
        if chart_ok:
            print(f"  ✅ JSON 解析成功，{len(charts)} 組圖表，series/x_labels 長度一致")
    except Exception as e:
        all_pass = False
        print(f"  ❌ JSON 解析失敗: {e}")

    # 4️⃣ HTML 結構
    print("\n── 4. HTML 結構 ──")
    html_ends = (html_content or '').rstrip().endswith('</html>')
    mail_ends = (mail_body or '').rstrip().endswith('</html>')
    mail_no_svg = '<svg' not in (mail_body or '')
    mail_no_script = '<script' not in (mail_body or '')
    if html_ends and mail_ends and mail_no_svg and mail_no_script:
        print("  ✅ 兩版均以 </html> 結尾；mail_body 無 <svg>/<script>")
    else:
        all_pass = False
        print(f"  ❌ html_ends={html_ends}, mail_ends={mail_ends}, no_svg={mail_no_svg}, no_script={mail_no_script}")

    # 5️⃣ insight 完整性 + 文字品質
    print("\n── 5. insight ──")
    broken_patterns = ['<b></b>', '<td class=i></td>', '<td class=i> </td>', '()</b>']
    found_broken = [p for p in broken_patterns if p in (mail_body or '')]
    if found_broken:
        warns.append(f"可能的殘破片段: {found_broken}")
    for m in re.finditer(r"<b>(AI 年度分析|AI 季度分析|趨勢判讀)：</b>(.*?)</td>", mail_body or ""):
        name, body = m.group(1), m.group(2)
        if "**" in body or re.search(r"(^|\s)#{1,6}\s", body):
            warns.append(f"{name} 含 markdown 記號")
        if re.match(r"\s*(?:[\w\s]*(?:摘要|分析|判讀)[:：])", body):
            warns.append(f"{name} 以標題開頭：{body[:30]}…")
        if "（AI 分析產生失敗）" in body or "（AI 分析尚未啟用）" in body:
            warns.append(f"{name} 無 LLM 內容")
        if "明年度" in body or "明年" in body:
            warns.append(f"{name} 出現「明年」字樣，請確認年度解讀：{body[:60]}…")
    if not warns:
        print("  ✅ insight 區塊結構完整、無 markdown / 標題重複")

    # 6️⃣ 內容關卡（品牌數、缺填、季度 '-'）
    print("\n── 6. 內容 ──")
    text = _text_of(mail_body)
    m_now = re.search(r"Major (\d+)", text)
    major_now = int(m_now.group(1)) if m_now else None
    if prev:
        m_prev = re.search(r"Major (\d+)", _text_of(prev["mail_body"]))
        if m_prev and major_now is not None and major_now < int(m_prev.group(1)):
            warns.append(f"品牌數 Major {major_now} 少於上期 {prev['brief_month']} 的 Major {m_prev.group(1)}")
    m_missing = re.search(r"尚未填報：([^。]+)。", text)
    if m_missing:
        warns.append(f"本期缺填品牌：{m_missing.group(1)}（沿用上期）")
    m_partial = re.search(r"未填齊季度以「-」表示、不計入 Major：([^。]+)。", text)
    if m_partial:
        warns.append(f"季度未填齊：{m_partial.group(1)}")
    print(f"  Major 列品牌數 = {major_now}")

    # ── 總結 ──
    print(f"\n{'='*60}")
    for w in warns:
        print(f"  ⚠️ {w}")
    if all_pass:
        print(f"✅ {APP} 格式關卡通過；內容警告 {len(warns)} 項" + ("，請人工確認後再寄信。" if warns else "。"))
    else:
        overall_pass = False
        print(f"❌ {APP} 有格式關卡未通過，請檢查上方錯誤。")
    print(f"{'='*60}")

print(f"\n{'━'*60}")
if overall_pass:
    print("🎉 全部應用格式關卡通過！（內容警告見各 app 區塊）")
else:
    print("⚠️ 部分應用驗證未通過，請往上檢查。")
print(f"{'━'*60}")


In [ ]:
# [c08] preview_mnt
# ── 預覽 MNT 最新三欄（一次複製3）──
import json

row = spark.sql("""
    SELECT html_content, mail_body, chart_data
    FROM micenter.mi3_datahub_prod.g_sales_brief_mail_report
    WHERE app_name = 'MNT'
    ORDER BY brief_month DESC
    LIMIT 1
""").collect()[0]

# ── 合併輸出（方便一次全部複製）──
SEP = "\n" + "═" * 80 + "\n"
output = []
output.append("═" * 80)
output.append("【1/3 html_content】")
output.append("═" * 80)
output.append(row["html_content"])
output.append(SEP)
output.append("【2/3 mail_body】")
output.append("═" * 80)
output.append(row["mail_body"])
output.append(SEP)
output.append("【3/3 chart_data】")
output.append("═" * 80)
output.append(json.dumps(json.loads(row["chart_data"]), indent=2, ensure_ascii=False))
output.append("\n" + "═" * 80)

print("\n".join(output))

# ── 瀏覽器預覽（mail_body）──
displayHTML(row["mail_body"])

In [ ]:
# [c09] preview_tv
# ── 預覽 TV 最新三欄（一次複製3）──
import json

row = spark.sql("""
    SELECT html_content, mail_body, chart_data
    FROM micenter.mi3_datahub_prod.g_sales_brief_mail_report
    WHERE app_name = 'TV'
    ORDER BY brief_month DESC
    LIMIT 1
""").collect()[0]

# ── 合併輸出（方便一次全部複製）──
SEP = "\n" + "═" * 80 + "\n"
output = []
output.append("═" * 80)
output.append("【1/3 html_content】")
output.append("═" * 80)
output.append(row["html_content"])
output.append(SEP)
output.append("【2/3 mail_body】")
output.append("═" * 80)
output.append(row["mail_body"])
output.append(SEP)
output.append("【3/3 chart_data】")
output.append("═" * 80)
output.append(json.dumps(json.loads(row["chart_data"]), indent=2, ensure_ascii=False))
output.append("\n" + "═" * 80)

print("\n".join(output))

# ── 瀏覽器預覽（mail_body）──
displayHTML(row["mail_body"])

In [ ]:
# [c10] preview_nb
# ── 預覽 NB 最新三欄（一次複製3）──
import json

row = spark.sql("""
    SELECT html_content, mail_body, chart_data
    FROM micenter.mi3_datahub_prod.g_sales_brief_mail_report
    WHERE app_name = 'NB'
    ORDER BY brief_month DESC
    LIMIT 1
""").collect()[0]

# ── 合併輸出（方便一次全部複製）──
SEP = "\n" + "═" * 80 + "\n"
output = []
output.append("═" * 80)
output.append("【1/3 html_content】")
output.append("═" * 80)
output.append(row["html_content"])
output.append(SEP)
output.append("【2/3 mail_body】")
output.append("═" * 80)
output.append(row["mail_body"])
output.append(SEP)
output.append("【3/3 chart_data】")
output.append("═" * 80)
output.append(json.dumps(json.loads(row["chart_data"]), indent=2, ensure_ascii=False))
output.append("\n" + "═" * 80)

print("\n".join(output))

# ── 瀏覽器預覽（mail_body）──
displayHTML(row["mail_body"])